# CICIoT-2023 20-Method Benchmark -- Full Run, 18 Files (v10)

Same code the smoke test just proved works end-to-end on your machine. This
version points at all 18 real files and uses the real training budget -- this
is the actual multi-day run, not a quick check.

**Edit only `DATA_PATH` in the config cell below** if your files aren't in the
folder already set. Everything else is pre-set correctly.

**Before you start:**
- Estimated total runtime: roughly 4.3 days if run straight through, sequentially
  -- but it doesn't have to be one sitting. `already_done()` checkpoints every
  (fold, model) pair to `per_fold_results.csv`, so you can stop and resume across
  multiple sessions without losing completed work.
- FedHAN alone accounts for over half that estimate. Nothing to do about it now
  unless you want to revisit that tradeoff.
- Output goes to a directory separate from the smoke test's, so nothing here can
  collide with or overwrite what you already have.

**Already fixed in this pipeline:**
- Feature filter now fits per fold, not globally (no leakage); CAFE-TF's groups
  are matched by feature name, not column position
- VAE Detector's stale-cache bug (`clear_cached_results()` available if you ever
  need to force a specific model to re-run)
- Anomaly-score calibration fit once on training data, not recomputed per batch
- CD-diagram's critical-value constant, silently wrong for >10 methods -- fixed
- FedHAN's gradient clipping: checked, was already correct, not a bug
- The read-only-array crash in the smoke test's subsampler doesn't apply here --
  that code isn't part of this file at all

**Still genuinely open:** CNN-1D's near-zero accuracy has no confirmed root
cause. Expect to see it in the results; it's a known, documented open question,
not a new failure.


In [ ]:
# =====================================================================================
# CONFIGURATION -- edit only DATA_PATH below if needed, then run all cells in order.
# =====================================================================================
import os, re, glob
import numpy as np

# ---- Path handling: works whether you paste a Windows path or a WSL/Linux path ----
def normalize_path(path):
    """C:\\Users\\... or C:/Users/... -> /mnt/c/Users/... automatically. Already-Unix
    paths pass through unchanged. This kernel runs under WSL2 (confirmed: platform.
    system()=='Linux', kernel string contains 'WSL2'), so a raw Windows path does NOT
    work here regardless of what the VS Code file-explorer sidebar shows -- this
    conversion is what makes that difference not matter."""
    m = re.match(r'^([A-Za-z]):[\\/](.*)$', path)
    if m:
        return f'/mnt/{m.group(1).lower()}/{m.group(2).replace(chr(92), "/")}'
    return path.replace(chr(92), '/') if chr(92) in path else path

import platform
_is_wsl = 'WSL' in platform.uname().release or 'microsoft' in platform.uname().release.lower()
print(f"Environment: {platform.system()} ({'WSL2' if _is_wsl else platform.uname().release}) -- "
      f"{'Windows-style paths below will auto-convert' if _is_wsl else 'running natively, paths used as given'}")

# ---- EDIT THIS if your files live somewhere else: matches all 18 real part-files ----
DATA_PATH = normalize_path(r"C:\Users\babar.hameed\my_project\riz\part-*-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv")
_matched = sorted(glob.glob(DATA_PATH))
print(f"DATA_PATH glob matches {len(_matched)} file(s)"
      + ("" if len(_matched) == 18 else f"  *** expected 18, check the path/pattern above ***"))

# ---- Output directory: separate from the smoke test's ./smoke_test_outputs, so
# nothing here can collide with or overwrite what you already ran --
OUTPUT_DIR = normalize_path("/home/babar/eval_outputs_full")
RESULTS_CSV = f"{OUTPUT_DIR}/per_fold_results.csv"
PROBA_DIR = f"{OUTPUT_DIR}/fold_probabilities"
LOSS_DIR = f"{OUTPUT_DIR}/loss_history"
for _d in (OUTPUT_DIR, PROBA_DIR, LOSS_DIR, f"{OUTPUT_DIR}/figures"):
    os.makedirs(_d, exist_ok=True)
print(f"Output to: {OUTPUT_DIR}")

# ---- Reproducibility / CV -- real run settings ----
SEED = 42
N_FOLDS = 5

# ---- Label grouping: 34 raw CICIoT-2023 labels -> 8 classes ----
LABEL_GROUP_RULES = {
    "BenignTraffic": "Normal", "MITM-ArpSpoofing": "Spoofing", "DNS_Spoofing": "Spoofing",
    "SqlInjection": "Web", "XSS": "Web", "CommandInjection": "Web",
    "BrowserHijacking": "Web", "Uploading_Attack": "Web",
}
LABEL_PREFIX_RULES = [("DDoS-", "DDoS"), ("DoS-", "DoS"), ("Mirai-", "Mirai"), ("Recon-", "Recon")]
LABEL_DEFAULT_GROUP = "BruteForce_Other"
CLASS_ORDER = ["DDoS", "DoS", "Mirai", "Normal", "Spoofing", "Recon", "BruteForce_Other", "Web"]
N_CLASSES = len(CLASS_ORDER)

# ---- Feature engineering ----
N_RAW_FEATURES = 46
CONST_VAR_THRESHOLD = 0.001
VIF_THRESHOLD = 10
N_FINAL_FEATURES = 26   # reference value; fit per-fold, see the per-fold [NOTE] if it differs

# ---- Class imbalance handling -- multipliers of a fold's own real count, verified
# against this exact 18-file dataset's actual class proportions earlier ----
IMBALANCE_CONFIG = dict(
    majority_class="DDoS", majority_cap_to="DoS",
    aggressive_smote_targets={"Recon": 6000/2366, "BruteForce_Other": 4500/400, "Web": 3500/161},
    smote_k_neighbors=5,
)

# ---- Baseline hyperparameters -- real epoch budgets ----
BASELINE_CONFIG = {
    "Logistic Regression": dict(C=1.0, penalty="l2", max_iter=2000),
    "Naive Bayes": dict(),
    "Linear SVM": dict(C=1.0, loss="hinge", max_iter=5000),
    "RBF-SVM": dict(C=1.0, gamma="scale"),
    "MLP": dict(hidden_dims=[256, 128, 64], dropout=0.3, epochs=40),
    "LSTM": dict(hidden=256, layers=2, dropout=0.3, epochs=25, bidirectional=False),
    "Bi-LSTM": dict(hidden=256, layers=2, dropout=0.3, epochs=25, bidirectional=True),
    "CNN-1D": dict(channels=[32, 64, 128], epochs=25),
    "Isolation Forest": dict(n_estimators=100),
    "One-Class SVM": dict(kernel="rbf"),
    "VAE Detector": dict(latent_dim=16, epochs=30),
}
# Kernel SVM training time scales ~quadratic-to-cubic in sample count. Full uncapped
# fold at this scale (~1.6M post-resampling rows) is estimated at 8-102 days for
# RBF-SVM alone -- infeasible regardless of time priority. 100,000 is a genuinely
# generous cap (~40min estimated) without that risk.
RBF_SVM_MAX_TRAIN = 100000

# ---- Novel architectures -- real epoch/round budgets ----
# PANE and ARID raised 40->100 to match CAFE-TF, removing the budget confound
# reviewers flagged repeatedly -- raising the weaker-budgeted models rather than
# cutting CAFE-TF down, since this run prioritizes result quality.
PANE_CONFIG = dict(gru_hidden=128, gru_layers=4, n_frequencies=16, epochs=100, lr=1e-3)
CAFE_TF_CONFIG = dict(d_model=128, n_heads=8, n_layers=4, d_ff=512,
                       feature_groups=[4, 1, 5, 4, 4, 2, 4, 2], epochs=100, warmup_epochs=10,
                       lr=3e-4, weight_decay=0.01)
ARID_CONFIG = dict(hidden_dims=[256, 512, 512, 256, 128], epochs=100, lr=1e-3,
                    pgd_epsilon=0.03, pgd_alpha=0.01, pgd_steps=7, adv_lambda=0.5)
FEDHAN_CONFIG = dict(cnn_channels=[32, 64, 128], kernel_size=3, fc_dims=[128, 64],
                      n_clients=10, dirichlet_alpha=0.5, local_epochs=10, comm_rounds=96,
                      beta_sharpness=2.0, dp_enabled=False, dp_sigma=1.1, dp_clip_norm=1.0, dp_delta=1e-5)
CASCADE_CONFIG = dict(tau1=0.85, tau2=0.80)

# ---- Literature baselines A1-A5 -- real epoch budgets ----
IAT_COLUMN_NAME = "IAT"
A1_TRANSFORMER_CONFIG = dict(d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.1,
                              epochs=25, lr=1e-3, batch_size=512)
A2_IAT_DNN_CONFIG = dict(epochs=15, lr=1e-3, batch_size=128)
A4_GIDS_CONFIG = dict(d_model=64, n_layers=2, k_neighbors=10, epochs=25, lr=1e-3, batch_size=512)
A5_GCN_DQN_CONFIG = dict(d_model=64, n_gcn_layers=2, n_heads=4, edge_percentile=0.1,
                          epochs=25, lr=1e-3, weight_decay=1e-2, batch_size=256,
                          gamma=0.99, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.97,
                          dqn_batch_size=32, target_sync_every=5)
RUN_A3_LLM_RAG = False
A3_LLM_RAG_CONFIG = dict(model_id="meta-llama/Llama-3.2-1B", lora_r=16, lora_alpha=32,
                          lora_dropout=0.1,
                          target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
                          max_seq_len=512, epochs=3, lr=5e-4, batch_size=4,
                          max_train_per_class=300, max_eval_per_class=60,
                          use_rag=True, rag_top_k_candidates=20, rag_top_n_context=3)

MODEL_ORDER = [
    "Logistic Regression", "Naive Bayes", "Linear SVM", "RBF-SVM",
    "MLP", "LSTM", "Bi-LSTM", "CNN-1D",
    "Isolation Forest", "One-Class SVM", "VAE Detector",
    "A1 Transformer", "A2 IAT-DNN", "A4 GIDS", "A5 GCN-DQN",
    *(["A3 LLM+RAG"] if RUN_A3_LLM_RAG else []),
    "PANE", "CAFE-TF", "ARID", "FedHAN", "Cascaded Ensemble",
]
print(f"Config loaded: {len(MODEL_ORDER)} models, {N_FOLDS}-fold CV (FULL RUN scale), seed={SEED}")


In [ ]:
#!pip install torch torchvision torchaudio xgboost lightgbm imbalanced-learn
#!pip install transformers peft bitsandbytes accelerate   # only needed if RUN_A3_LLM_RAG = True

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:
# =====================================================================================
# Imports & Reproducibility Seeding
# =====================================================================================
import random, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # v8: moved here from Cell 45 -- Section 1 (Dataset
                                  # Exploration) needs plt earlier than that cell runs
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC, SVC, OneClassSVM
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               IsolationForest)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print("Imports OK. Device:", DEVICE)


## Label Grouping, Global Feature Filtering, Class Weights, Resampling

In [ ]:
import glob
# =====================================================================================
# CELL 4 -- Dataset Loading, Label Grouping, Global Feature Filtering (v2)
# =====================================================================================

LABEL_CANDIDATES = ["label", "Label", "class", "Class", "attack_type", "Attack_type"]


def group_label(raw_label):
    """34 raw CICIoT-2023 labels -> 8 classes, per Section III-A."""
    if raw_label in LABEL_GROUP_RULES:
        return LABEL_GROUP_RULES[raw_label]
    for prefix, group in LABEL_PREFIX_RULES:
        if raw_label.startswith(prefix):
            return group
    return LABEL_DEFAULT_GROUP


def _read_one_part(path):
    """Read a single file, sniffing CSV vs Parquet from the extension, falling back
    to CSV (Spark part-files are frequently extensionless or end .c000)."""
    lower = path.lower()
    if lower.endswith((".parquet", ".pq")) or ".snappy.parquet" in lower:
        return pd.read_parquet(path)
    try:
        return pd.read_csv(path)
    except Exception as e:
        try:
            return pd.read_parquet(path)
        except Exception:
            raise e


def load_raw_dataset(path):
    """v9: `path` may be a single file, a glob pattern (e.g.
    'data/part-*-c000'), or a list/tuple of explicit paths -- this is what lets the
    4M-row, 18-part-file run use the exact same function as the original 348k
    single-CSV run. Every part is read and schema-checked (identical columns, same
    label column) before concatenation; a schema mismatch raises immediately rather
    than silently producing NaNs or a misaligned label column partway through a
    multi-hour run.
    """
    if isinstance(path, (list, tuple)):
        paths = list(path)
    elif any(ch in str(path) for ch in "*?["):
        paths = sorted(glob.glob(path))
        if not paths:
            raise FileNotFoundError(f"No files matched glob pattern: {path}")
    else:
        paths = [path]

    frames = []
    ref_cols = None
    for p in paths:
        df_part = _read_one_part(p)
        if ref_cols is None:
            ref_cols = list(df_part.columns)
        elif list(df_part.columns) != ref_cols:
            raise ValueError(
                f"Schema mismatch: {p} has columns {list(df_part.columns)[:5]}... "
                f"but the first part had {ref_cols[:5]}... -- every part file must "
                f"share identical columns before concatenation."
            )
        frames.append(df_part)
        print(f"  read part {os.path.basename(p)}: {df_part.shape[0]:,} rows")
    df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]

    label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), df.columns[-1])

    # v9 BUGFIX: rows with a missing/blank label crashed group_label() with
    # AttributeError ('float' object has no attribute 'startswith'), found by
    # testing against a real sample file where ~99.8% of rows had a genuinely
    # blank label column. .astype(str) does NOT turn NaN into the string "nan"
    # under pandas' newer nullable string dtype -- it stays a real float NaN,
    # which group_label()'s .startswith() call cannot handle. These rows carry
    # no usable supervision signal regardless, so they are dropped explicitly,
    # loudly, and before any grouping is attempted -- not silently absorbed into
    # a default class, and not left to crash partway through a multi-hour run.
    missing_label_mask = df[label_col].isna()
    n_missing = int(missing_label_mask.sum())
    if n_missing:
        pct = 100 * n_missing / len(df)
        print(f"  [WARNING] {n_missing:,}/{len(df):,} rows ({pct:.1f}%) have a missing "
              f"label and are being DROPPED before training. If this percentage is "
              f"unexpectedly high, check the source file before proceeding.")
        df = df.loc[~missing_label_mask].reset_index(drop=True)

    y_raw = df[label_col].astype(str)
    already_grouped = set(y_raw.unique()) <= set(CLASS_ORDER)
    y_grouped = y_raw if already_grouped else y_raw.map(group_label)
    X = df.drop(columns=[label_col])
    X = X.select_dtypes(include=[np.number]).copy()
    print(f"Loaded {len(paths)} part file(s), {X.shape[0]:,} usable rows "
          f"({n_missing:,} dropped for missing labels), "
          f"{X.shape[1]} raw numeric columns, label column = '{label_col}'"
          + ("" if already_grouped else f" (grouped {y_raw.nunique()} raw labels -> {y_grouped.nunique()} classes)"))
    return X, y_grouped


def encode_labels(y_grouped):
    class_to_idx = {c: i for i, c in enumerate(CLASS_ORDER)}
    missing = set(y_grouped.unique()) - set(CLASS_ORDER)
    assert not missing, f"Unexpected classes after grouping: {missing}"
    y = y_grouped.map(class_to_idx).values.astype(np.int64)
    return y, CLASS_ORDER, class_to_idx


# ---- v9: feature filtering fit PER FOLD (training split only), not once globally --
# Fixes the leakage flagged in Section VI: the filter no longer sees any fold's test
# rows. Because the retained feature SET can now vary slightly fold to fold, CAFE-TF's
# group structure is looked up by FEATURE NAME (Table II) rather than assumed to sit
# at fixed column positions -- see reorder_columns_by_group() below.

FEATURE_GROUP_NAMES = {
    1: ["flow_duration", "Header_Length", "Protocol Type", "Duration"],
    2: ["Rate"],
    3: ["fin_flag_number", "syn_flag_number", "psh_flag_number", "ack_flag_number", "syn_count"],
    4: ["fin_count", "urg_count", "rst_count", "HTTP"],
    5: ["HTTPS", "TCP", "UDP", "ICMP"],
    6: ["Tot sum", "Min"],
    7: ["Max", "AVG", "Tot size", "IAT"],
    8: ["Covariance", "Variance"],
}
ALL_GROUPED_NAMES = [n for g in FEATURE_GROUP_NAMES.values() for n in g]


def fit_fold_feature_filter(X_train, const_thresh=CONST_VAR_THRESHOLD, vif_thresh=VIF_THRESHOLD):
    """Constant-variance + collinearity filter fit on ONE FOLD's training split only."""
    variances = X_train.var()
    keep = variances[variances > const_thresh].index.tolist()
    Xk = X_train[keep]
    corr = Xk.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > 0.95)]
    return [c for c in keep if c not in to_drop]


def reorder_columns_by_group(cols_present, group_names=FEATURE_GROUP_NAMES):
    """Regroup a fold's surviving columns into CAFE-TF's contiguous group order by
    NAME, so torch.split() on the resulting tensor always cuts at true group
    boundaries regardless of which borderline features this fold's filter kept.
    Returns (reordered_cols, group_sizes) -- group_sizes sums to len(cols_present);
    an entirely unrecognized surviving column is appended as its own trailing group
    rather than silently dropped, and a group that loses all its members that fold
    is omitted rather than passed to nn.Linear(0, ...)."""
    present = set(cols_present)
    reordered, group_sizes = [], []
    for g in sorted(group_names):
        members = [f for f in group_names[g] if f in present]
        if members:
            reordered.extend(members)
            group_sizes.append(len(members))
    ungrouped = [c for c in cols_present if c not in ALL_GROUPED_NAMES]
    if ungrouped:
        reordered.extend(ungrouped)
        group_sizes.append(len(ungrouped))
    assert len(reordered) == len(cols_present), (
        f"reorder_columns_by_group dropped/duplicated columns: "
        f"{len(cols_present)} in, {len(reordered)} out"
    )
    return reordered, group_sizes


def apply_feature_filter(X, cols):
    return X[cols]


# ---- Class weights: sqrt-inverse-frequency on the POST-resampling distribution
# (Section III-D), gentler than raw inverse-frequency so an already-aggressively-SMOTEd
# minority isn't double-corrected.
def compute_class_weights(y_resampled, n_classes=N_CLASSES):
    counts = np.bincount(y_resampled, minlength=n_classes).astype(float)
    counts = np.clip(counts, 1, None)
    N = counts.sum()
    w = np.sqrt(N / (n_classes * counts))
    return w  # index-aligned with CLASS_ORDER


# ---- Fold-internal resampling: DDoS capped to DoS's count (this fold), Recon/
# Brute-Force-Other/Web SMOTE'd to fold-relative multipliers of their own real count
# (Section III-D). Everything else is left at its natural post-undersampling count.
def resample_fold(X_train, y_train, class_to_idx, cfg=IMBALANCE_CONFIG):
    counts = pd.Series(y_train).value_counts()
    idx_of = lambda name: class_to_idx[name]

    # 1) undersample the majority class to match the cap-target class's count
    maj_idx = idx_of(cfg["majority_class"])
    cap_idx = idx_of(cfg["majority_cap_to"])
    if maj_idx in counts.index and cap_idx in counts.index and counts[maj_idx] > counts[cap_idx]:
        under_strategy = {maj_idx: int(counts[cap_idx])}
        rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=SEED)
        X_train, y_train = rus.fit_resample(X_train, y_train)

    # 2) aggressive SMOTE for the severe-minority classes, target = multiplier x
    #    THIS FOLD's own real count for that class (matches Section III-D's ratios
    #    regardless of whether this fold is the full 348k or a smaller sample)
    counts = pd.Series(y_train).value_counts()
    over_strategy = {}
    for cls_name, multiplier in cfg["aggressive_smote_targets"].items():
        c_idx = idx_of(cls_name)
        if c_idx not in counts.index:
            continue
        real_n = int(counts[c_idx])
        target_n = max(real_n, int(round(real_n * multiplier)))
        k_neighbors = min(cfg["smote_k_neighbors"], max(1, real_n - 1))
        over_strategy[c_idx] = (target_n, k_neighbors)

    for c_idx, (target_n, k_neighbors) in over_strategy.items():
        try:
            sm = SMOTE(sampling_strategy={c_idx: target_n}, random_state=SEED, k_neighbors=k_neighbors)
            X_train, y_train = sm.fit_resample(X_train, y_train)
        except ValueError as e:
            print(f"  [SMOTE skipped for class {c_idx}: {e}]")

    return X_train, y_train


def scale_fold(X_train, X_test):
    scaler = RobustScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)
    return Xtr, Xte, scaler


print("Data-loading, label-grouping, global-filter, and class-weighted resampling functions defined.")


## Dataset Exploration (v8)

Runs independently of the 5-fold evaluation loop below -- loads the full dataset once, applies the same label-regrouping and global feature-filtering functions defined above, and produces the exploratory plots in this section. Safe to run on its own before committing to the full `run_full_evaluation()` run further down; nothing in this section fits or evaluates a model.

In [ ]:
# =====================================================================================
# Dataset Exploration: load once for visualization
# =====================================================================================
# BUGFIX: DATA_PATH is now a glob PATTERN (matches all 18 files), not a single literal
# file -- pd.read_csv(DATA_PATH) can't open a pattern directly. Reusing the real,
# already-glob-aware load_raw_dataset() here fixes that; pointed at just the first
# matched file (not all 18) since this cell is a quick illustrative preview, not the
# rigorous evaluation -- one file's ~230k+ rows is already far more than a class
# distribution or correlation plot needs, and loading all 18 (~4.26M rows) here would
# cost real time and memory for no benefit to what this cell actually produces.
X_raw_viz, y_grouped_viz = load_raw_dataset(_matched[0])

# BUGFIX: Fig E1 below needs the TRUE pre-grouping label distribution (up to 34
# distinct raw labels) side by side with the regrouped 8-class one -- y_grouped_viz
# alone can't provide that (it's already grouped). load_raw_dataset() doesn't return
# the raw series separately, so read just that one column again here -- cheap (one
# column, not the other 46 features) and reuses the exact same label-column detection
# load_raw_dataset() itself uses, so this can't disagree with what it found.
_label_col_viz = next((c for c in LABEL_CANDIDATES if c in pd.read_csv(_matched[0], nrows=0).columns),
                       pd.read_csv(_matched[0], nrows=0).columns[-1])
y_raw_viz = pd.read_csv(_matched[0], usecols=[_label_col_viz])[_label_col_viz].astype(str)

y_viz, _, class_to_idx_viz = encode_labels(y_grouped_viz)
FEATURE_COLS_VIZ = fit_fold_feature_filter(X_raw_viz)  # global fit is fine here -- exploratory plotting only, no train/test split involved
X_final_viz = apply_feature_filter(X_raw_viz, FEATURE_COLS_VIZ)

FIG_DIR = f"{OUTPUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Dataset ready for exploration: {X_final_viz.shape[0]:,} rows, "
      f"{X_final_viz.shape[1]} features, {y_raw_viz.nunique()} raw labels -> "
      f"{y_grouped_viz.nunique()} regrouped classes.")


In [ ]:
# ---- Fig E1: class distribution, raw labels vs regrouped 8-class taxonomy ----
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

raw_counts = y_raw_viz.value_counts().sort_values(ascending=True)
axes[0].barh(raw_counts.index.astype(str), raw_counts.values, color="#8C9BAB",
             edgecolor="black", linewidth=0.4)
axes[0].set_xscale("log")
axes[0].set_xlabel("Count (log scale)")
axes[0].set_title(f"Raw CICIoT-2023 Labels (n={y_raw_viz.nunique()})", fontsize=11, fontweight="bold")
axes[0].tick_params(axis="y", labelsize=7)

grp_counts = pd.Series(y_grouped_viz).value_counts().reindex(CLASS_ORDER)
colors_grp = plt.cm.tab10(np.linspace(0, 1, len(CLASS_ORDER)))
axes[1].bar(grp_counts.index, grp_counts.values, color=colors_grp, edgecolor="black", linewidth=0.6)
axes[1].set_yscale("log")
axes[1].set_ylabel("Count (log scale)")
axes[1].set_title(f"Regrouped 8-Class Taxonomy (imbalance ratio "
                   f"{grp_counts.max()/grp_counts.min():.0f}:1)", fontsize=11, fontweight="bold")
axes[1].tick_params(axis="x", rotation=40)
for i, v in enumerate(grp_counts.values):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
fig_path = f"{FIG_DIR}/DatasetExploration_class_distribution.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig E2: feature correlation heatmap (final features), CAFE-TF groups annotated ----
corr_viz = X_final_viz.corr()

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(corr_viz.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(FEATURE_COLS_VIZ))); ax.set_xticklabels(FEATURE_COLS_VIZ, rotation=90, fontsize=6)
ax.set_yticks(range(len(FEATURE_COLS_VIZ))); ax.set_yticklabels(FEATURE_COLS_VIZ, fontsize=6)
plt.colorbar(im, ax=ax, label="Pearson r", shrink=0.8)

group_sizes = CAFE_TF_CONFIG.get("feature_groups", [])
if sum(group_sizes) == len(FEATURE_COLS_VIZ):
    boundary = -0.5
    for g in group_sizes[:-1]:
        boundary += g
        ax.axhline(boundary, color="black", linewidth=1.2)
        ax.axvline(boundary, color="black", linewidth=1.2)
    ax.set_title(f"Feature Correlation ({len(FEATURE_COLS_VIZ)} features; black lines = "
                 f"CAFE-TF's {len(group_sizes)} semantic groups)", fontsize=10, fontweight="bold")
else:
    ax.set_title(f"Feature Correlation ({len(FEATURE_COLS_VIZ)} features)", fontsize=10, fontweight="bold")
    print(f"  [note] {len(FEATURE_COLS_VIZ)} final features != sum(CAFE_TF_CONFIG['feature_groups'])"
          f"={sum(group_sizes)} -- group boundaries not drawn. Update CAFE_TF_CONFIG if this "
          f"persists on the full dataset.")

plt.tight_layout()
fig_path = f"{FIG_DIR}/DatasetExploration_correlation_heatmap.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig E3: 2D projection of scaled feature space, colored by class ----
# Per-class capped sample (not pure random) so minority classes (e.g. Web, n=161 on the
# full dataset) are fully visible rather than nearly absent at a 5-6% random sampling rate.
_max_per_class = 2500
_sub_parts = []
for c_idx in range(N_CLASSES):
    idx_c = X_final_viz.index[y_viz == c_idx]
    n_take = min(_max_per_class, len(idx_c))
    _sub_parts.append(pd.Series(idx_c).sample(n=n_take, random_state=SEED))
_sub_idx = pd.concat(_sub_parts).values
X_sub = X_final_viz.loc[_sub_idx]
y_sub = pd.Series(y_viz, index=X_final_viz.index).loc[_sub_idx]
X_sub_scaled = RobustScaler().fit_transform(X_sub)

try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=SEED, n_jobs=1)
    proj = reducer.fit_transform(X_sub_scaled)
    proj_name = "UMAP"
except ImportError:
    from sklearn.decomposition import PCA
    proj = PCA(n_components=2, random_state=SEED).fit_transform(X_sub_scaled)
    proj_name = "PCA"
    print("  [note] umap-learn not installed -- falling back to PCA. "
          "`pip install umap-learn` for a UMAP projection instead.")

fig, ax = plt.subplots(figsize=(8, 7))
cmap = plt.cm.tab10(np.linspace(0, 1, len(CLASS_ORDER)))
for c_idx, c_name in enumerate(CLASS_ORDER):
    mask = (y_sub.values == c_idx)
    ax.scatter(proj[mask, 0], proj[mask, 1], s=4, alpha=0.5, color=cmap[c_idx], label=c_name)
ax.set_title(f"{proj_name} Projection of Scaled Feature Space\n"
             f"(n={len(_sub_idx):,}, capped at {_max_per_class}/class)", fontsize=10, fontweight="bold")
ax.legend(markerscale=3, fontsize=8, loc="best")
plt.tight_layout()
fig_path = f"{FIG_DIR}/DatasetExploration_{proj_name.lower()}_projection.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig E4: fold-internal resampling effect (before -> after), one illustrative fold ----
from sklearn.model_selection import StratifiedKFold as _SKF_viz

_skf_viz = _SKF_viz(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
_train_idx_viz, _test_idx_viz = next(iter(_skf_viz.split(X_final_viz, y_viz)))
X_train_viz_raw = X_final_viz.iloc[_train_idx_viz].values
y_train_viz_raw = y_viz[_train_idx_viz]

before_counts = pd.Series(y_train_viz_raw).value_counts().reindex(range(N_CLASSES)).fillna(0)
X_res_viz, y_res_viz = resample_fold(X_train_viz_raw, y_train_viz_raw, class_to_idx_viz)
after_counts = pd.Series(y_res_viz).value_counts().reindex(range(N_CLASSES)).fillna(0)

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(N_CLASSES)
w = 0.35
ax.bar(x - w/2, before_counts.values, width=w, label="Before resampling",
       color="#8C9BAB", edgecolor="black", linewidth=0.5)
ax.bar(x + w/2, after_counts.values, width=w, label="After resampling",
       color="#2E7D5B", edgecolor="black", linewidth=0.5)
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(CLASS_ORDER, rotation=30)
ax.set_ylabel("Count (log scale, training split only)")
ax.set_title("Fold-Internal Resampling Effect (fold 1 of the seeded StratifiedKFold split)",
             fontsize=10, fontweight="bold")
ax.legend()
plt.tight_layout()
fig_path = f"{FIG_DIR}/DatasetExploration_resampling_effect.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")
del X_train_viz_raw, y_train_viz_raw, X_res_viz, y_res_viz


In [ ]:
# ---- Fig E5: stratification check -- class proportion per fold's test split ----
_skf_check = _SKF_viz(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_props = []
for fold_i, (tr_i, te_i) in enumerate(_skf_check.split(X_final_viz, y_viz), start=1):
    prop = pd.Series(y_viz[te_i]).value_counts(normalize=True).reindex(range(N_CLASSES)).fillna(0)
    fold_props.append(prop.values)
fold_props = np.array(fold_props)  # (N_FOLDS, N_CLASSES)

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(N_CLASSES)
w = 0.8 / N_FOLDS
for f in range(N_FOLDS):
    ax.bar(x + (f - N_FOLDS/2)*w + w/2, fold_props[f], width=w, label=f"Fold {f+1}")
ax.set_xticks(x); ax.set_xticklabels(CLASS_ORDER, rotation=30)
ax.set_ylabel("Proportion of test fold")
ax.set_title("Stratification Check: Class Proportion per Fold's Test Split\n"
             "(bars should look nearly identical across folds if StratifiedKFold worked as intended)",
             fontsize=10, fontweight="bold")
ax.legend(fontsize=7, ncol=5)
plt.tight_layout()
fig_path = f"{FIG_DIR}/DatasetExploration_fold_stratification_check.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")
print("Per-class std of test-fold proportion across the 5 folds (should be small):")
print(pd.Series(fold_props.std(axis=0), index=CLASS_ORDER).round(5))


## Shared Evaluation & Class-Weight-Aware PyTorch Training Helpers

In [ ]:
# =====================================================================================
# CELL 7 -- Evaluation Helpers (v2: class-weight-aware)
# =====================================================================================

def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")
    return dict(accuracy=acc, precision=prec, recall=rec, f1_macro=f1, auc=auc)


class TorchClassifierWrapper:
    """sklearn-like fit/predict_proba wrapper. Accepts an optional `class_weight`
    (Section III-D sqrt-inverse-frequency vector, index-aligned with CLASS_ORDER) and
    applies it as weighted cross-entropy, matching every neural model in the paper."""

    def __init__(self, model_fn, epochs=30, lr=1e-3, batch_size=512, n_classes=N_CLASSES):
        self.model_fn = model_fn
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.n_classes = n_classes
        self.model = None

    def fit(self, X, y, class_weight=None):
        n_features = X.shape[1]
        self.model = self.model_fn(n_features, self.n_classes).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)

        w = None
        if class_weight is not None:
            w = torch.tensor(class_weight, dtype=torch.float32).to(DEVICE)

        Xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.long)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=self.batch_size, shuffle=True)

        self.model.train()
        self.loss_history = []          # BUGFIX (v6): expose convergence behaviour
        for epoch in range(self.epochs):
            epoch_loss, n_batches = 0.0, 0
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                logits = self.model(xb)
                loss = F.cross_entropy(logits, yb, weight=w)
                loss.backward()
                opt.step()
                epoch_loss += loss.item()
                n_batches += 1
            sched.step()
            self.loss_history.append(epoch_loss / max(1, n_batches))
        return self

    @torch.no_grad()
    def predict_proba(self, X):
        self.model.eval()
        Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        probs = []
        for i in range(0, len(Xt), 4096):
            batch = Xt[i:i + 4096]
            logits = self.model(batch)
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        return np.concatenate(probs, axis=0)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)




# ---------------------------------------------------------------------------------
# BUGFIX (v6): conv output-size inference without corrupting BatchNorm statistics.
#
# The v2 pipeline inferred the flatten dimension by pushing a torch.zeros dummy
# tensor through the conv stack. Because a freshly-constructed nn.Module is in
# TRAINING mode, every BatchNorm1d layer in that stack updated its running_mean /
# running_var from an all-zero batch before any real data was ever seen. This helper
# performs the same shape inference in eval() mode -- where BatchNorm reads its
# running statistics but does not update them -- and restores the previous mode.
# ---------------------------------------------------------------------------------
def infer_conv_flat_dim(conv_module, in_dim):
    was_training = conv_module.training
    conv_module.eval()
    try:
        with torch.no_grad():
            flat_dim = conv_module(torch.zeros(1, 1, in_dim)).flatten(1).shape[1]
    finally:
        conv_module.train(was_training)
    return flat_dim


# ---------------------------------------------------------------------------------
# BUGFIX (v6): pooling that adapts to low-dimensional tabular input.
#
# With 26 input features, three successive MaxPool1d(2) stages collapse the feature
# axis to 3 positions (26 -> 13 -> 6 -> 3), discarding most of the signal before the
# classifier head ever sees it. This helper only pools while enough positions remain,
# so the same channel configuration works on 26 features as on a longer input.
# ---------------------------------------------------------------------------------
MIN_LEN_TO_POOL = 8

def build_conv_stack(channels, in_dim, kernel_size=3):
    layers, prev_c, cur_len = [], 1, in_dim
    for c in channels:
        layers += [nn.Conv1d(prev_c, c, kernel_size, padding=kernel_size // 2),
                   nn.BatchNorm1d(c), nn.ReLU()]
        if cur_len >= MIN_LEN_TO_POOL:
            layers.append(nn.MaxPool1d(2))
            cur_len //= 2
        prev_c = c
    return nn.Sequential(*layers)


print("Evaluation + class-weight-aware Torch wrapper helpers defined.")


## Baseline Models (class_weight-aware)

In [ ]:
# =====================================================================================
# Baseline Models (class_weight-aware)
# =====================================================================================
# Design note on the three anomaly-detection baselines (Isolation Forest, One-Class SVM,
# VAE Detector): one-class models fit on Normal-only traffic; P(Normal) comes from the
# calibrated anomaly score, remaining mass split uniformly across the 7 attack classes.
# Class weighting does not apply to these (they never see attack-class labels during
# fitting), matching the manuscript's scope ("neural architectures" + "tree-based and
# gradient-boosted baselines" -- the anomaly detectors are neither).

class MLPNet(nn.Module):
    def __init__(self, in_dim, n_classes, hidden_dims, dropout):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class SeqRNN(nn.Module):
    def __init__(self, in_dim, n_classes, hidden, layers, dropout, bidirectional):
        super().__init__()
        self.rnn = nn.LSTM(input_size=in_dim, hidden_size=hidden, num_layers=layers,
                            batch_first=True, dropout=dropout if layers > 1 else 0,
                            bidirectional=bidirectional)
        out_dim = hidden * (2 if bidirectional else 1)
        self.head = nn.Linear(out_dim, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :])


class CNN1D(nn.Module):
    def __init__(self, in_dim, n_classes, channels):
        super().__init__()
        # BUGFIX (v6): adaptive pooling + BN-safe shape inference (see Cell 7).
        self.conv = build_conv_stack(channels, in_dim, kernel_size=3)
        flat_dim = infer_conv_flat_dim(self.conv, in_dim)
        self.head = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        return self.head(x.flatten(1))


class WeightedSklearnWrapper:
    """Wraps any sklearn-style estimator so it shares the same
    fit(X, y, class_weight=...) interface as TorchClassifierWrapper. class_weight
    (a CLASS_ORDER-aligned vector) is converted to per-sample weights.

    v9: optional max_train_samples caps the TRAINING set size via stratified
    subsampling before fit() -- added because kernel SVM training time scales
    roughly quadratic-to-cubic in sample count, so moving from 348k to ~4M rows
    (~11.5x) risks turning RBF-SVM's already-slow 4851s/fold into days per fold,
    not an ~11.5x slowdown. Evaluation is unaffected: predict/predict_proba still
    run on the fold's FULL, untouched test split, so reported metrics remain
    comparable across dataset sizes; only the training budget is capped."""

    def __init__(self, estimator_fn, max_train_samples=None):
        self.estimator_fn = estimator_fn
        self.model = None
        self.max_train_samples = max_train_samples

    def fit(self, X, y, class_weight=None):
        self.model = self.estimator_fn()
        if self.max_train_samples is not None and len(y) > self.max_train_samples:
            from sklearn.model_selection import StratifiedShuffleSplit
            splitter = StratifiedShuffleSplit(n_splits=1, train_size=self.max_train_samples,
                                               random_state=SEED)
            keep_idx, _ = next(splitter.split(X, y))
            print(f"    [subsampled for tractability: {len(y):,} -> "
                  f"{len(keep_idx):,} training rows, stratified]")
            X = X.iloc[keep_idx] if hasattr(X, "iloc") else X[keep_idx]
            y = y[keep_idx]
        if class_weight is not None:
            sample_weight = np.asarray(class_weight)[y]
            self.model.fit(X, y, sample_weight=sample_weight)
        else:
            self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def predict(self, X):
        return self.model.predict(X)


class OneClassProbaWrapper:
    """One-class anomaly detectors -- class_weight is accepted for interface
    compatibility with the CV harness but intentionally ignored (see note above)."""
    def __init__(self, kind, n_classes=N_CLASSES, **kwargs):
        self.kind = kind
        self.n_classes = n_classes
        self.kwargs = kwargs
        self.model = None
        self.normal_class = None
        self._raw_min = None   # v9: fit-time calibration anchors (see below)
        self._raw_max = None

    def _raw_scores(self, X):
        if self.kind == "isolation_forest":
            return self.model.score_samples(X)
        elif self.kind == "ocsvm":
            return self.model.decision_function(X)
        else:
            return -_vae_reconstruction_error(self.model, X)

    def fit(self, X, y, class_weight=None):
        # BUGFIX (v7, changelog B.3): normal_class now resolves explicitly to
        # CLASS_ORDER's "Normal" index, matching this wrapper's own documented design
        # intent ("fit on Normal-only traffic"). Previously this was
        # `int(pd.Series(y).mode()[0])` -- the modal class of the RESAMPLED fold's
        # labels. Because fold-internal resampling caps DDoS down to DoS's count
        # (Section III-D), that modal class was not reliably "Normal"; it plausibly
        # resolved to DDoS/DoS instead. The diagnostic print below shows directly
        # whether that was firing on your data.
        self.normal_class = CLASS_ORDER.index("Normal")
        mode_class = int(pd.Series(y).mode()[0])
        if mode_class != self.normal_class:
            print(f"      [v7 diagnostic] OneClassProbaWrapper({self.kind}): resampled-fold "
                  f"mode class was '{CLASS_ORDER[mode_class]}' (idx {mode_class}), not 'Normal' "
                  f"(idx {self.normal_class}) -- confirms the v6 behaviour this fix corrects.")
        X_normal = X[y == self.normal_class]
        if self.kind == "isolation_forest":
            self.model = IsolationForest(random_state=SEED, **self.kwargs).fit(X_normal)
        elif self.kind == "ocsvm":
            sub = X_normal[np.random.RandomState(SEED).choice(len(X_normal),
                            size=min(5000, len(X_normal)), replace=False)]
            self.model = OneClassSVM(**self.kwargs).fit(sub)
        elif self.kind == "vae":
            self.model = _fit_small_vae(X_normal, self.kwargs.get("latent_dim", 16),
                                         self.kwargs.get("epochs", 30))

        # v9 BUGFIX: calibrate the min-max anchors ONCE here, on the fitted TRAINING
        # (Normal-only) scores -- exactly the same fit-train/apply-test discipline
        # RobustScaler already uses elsewhere in this pipeline. Previously
        # _anomaly_to_p_normal() recomputed min/max from whichever batch was passed
        # to predict_proba(), which is circular: it reports each point's anomaly
        # rank *within that specific test batch*, not anything about the point in
        # an absolute sense, so most points in most batches ended up pushed above
        # the 1/K arg-max threshold regardless of whether they were actually the
        # batch's real anomalies. This is the deeper miscalibration documented in
        # Section V-A; it is not automatically fixed by the normal_class correction
        # above, which is why both are needed together.
        train_raw = self._raw_scores(X_normal)
        self._raw_min = float(train_raw.min())
        self._raw_max = float(train_raw.max())
        return self

    def _anomaly_to_p_normal(self, X):
        raw = self._raw_scores(X)
        r = (raw - self._raw_min) / (self._raw_max - self._raw_min + 1e-9)
        return np.clip(r, 0.0, 1.0)   # test scores can fall outside the train-time range

    def predict_proba(self, X):
        p_normal = self._anomaly_to_p_normal(X)
        n = len(X)
        proba = np.zeros((n, self.n_classes))
        proba[:, self.normal_class] = p_normal
        other_classes = [c for c in range(self.n_classes) if c != self.normal_class]
        remaining = (1 - p_normal) / len(other_classes)
        for c in other_classes:
            proba[:, c] = remaining
        return proba

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


class _TinyVAE(nn.Module):
    def __init__(self, in_dim, latent_dim):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU())
        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, in_dim))

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)
        return self.dec(z), mu, logvar


def _fit_small_vae(X, latent_dim, epochs):
    model = _TinyVAE(X.shape[1], latent_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Xt = torch.tensor(X, dtype=torch.float32)
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=True)
    model.train()
    for _ in range(epochs):
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            recon, mu, logvar = model(xb)
            loss = F.mse_loss(recon, xb) - 0.001 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    return model


@torch.no_grad()
def _vae_reconstruction_error(model, X):
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    recon, _, _ = model(Xt)
    return ((Xt - recon) ** 2).mean(dim=1).cpu().numpy()


def build_baseline_models():
    cfg = BASELINE_CONFIG
    return {
        #"Decision Tree": WeightedSklearnWrapper(lambda: DecisionTreeClassifier(random_state=SEED, **cfg["Decision Tree"])),
        "Logistic Regression": WeightedSklearnWrapper(lambda: LogisticRegression(random_state=SEED, n_jobs=-1, **cfg["Logistic Regression"])),
        "Naive Bayes": WeightedSklearnWrapper(lambda: GaussianNB(**cfg["Naive Bayes"])),
        "Linear SVM": WeightedSklearnWrapper(lambda: CalibratedClassifierCV(LinearSVC(random_state=SEED, **cfg["Linear SVM"]), cv=3)),
        "RBF-SVM": WeightedSklearnWrapper(lambda: SVC(probability=True, random_state=SEED, **cfg["RBF-SVM"]), max_train_samples=RBF_SVM_MAX_TRAIN),

        "MLP": TorchClassifierWrapper(
            lambda d, c: MLPNet(d, c, cfg["MLP"]["hidden_dims"], cfg["MLP"]["dropout"]),
            epochs=cfg["MLP"]["epochs"]),
        "LSTM": TorchClassifierWrapper(
            lambda d, c: SeqRNN(d, c, cfg["LSTM"]["hidden"], cfg["LSTM"]["layers"],
                                 cfg["LSTM"]["dropout"], bidirectional=False),
            epochs=cfg["LSTM"]["epochs"]),
        "Bi-LSTM": TorchClassifierWrapper(
            lambda d, c: SeqRNN(d, c, cfg["Bi-LSTM"]["hidden"], cfg["Bi-LSTM"]["layers"],
                                 cfg["Bi-LSTM"]["dropout"], bidirectional=True),
            epochs=cfg["Bi-LSTM"]["epochs"]),
        "CNN-1D": TorchClassifierWrapper(
            lambda d, c: CNN1D(d, c, cfg["CNN-1D"]["channels"]), epochs=cfg["CNN-1D"]["epochs"]),
        "Isolation Forest": OneClassProbaWrapper("isolation_forest", **cfg["Isolation Forest"]),
        "One-Class SVM": OneClassProbaWrapper("ocsvm", **cfg["One-Class SVM"]),
        "VAE Detector": OneClassProbaWrapper("vae", **cfg["VAE Detector"]),
    }

print(f"{len(build_baseline_models())} baseline models are defined (v3, class_weight-aware). "
      f"Decision Tree exists in BASELINE_CONFIG but is commented out above and does not run.")


## Literature Baselines A1-A5 (external papers, added for comparison)

Five baselines from the outside literature, added alongside the baseline models above so PANE/CAFE-TF/ARID/FedHAN/the cascade have specific recent published methods to compare against, not just generic ML/DL baselines. **Citation status varies** -- A2, A3, and A5 are verified against the actual paper (full text read and cited below); A1 and A4 are best-effort placeholders because I could not verify those two citations by search -- see each cell's opening comment for exactly what was and wasn't confirmed, and CONFIRM-ME if you have the real sources.


## A1: Transformer, No Imbalance Handling (citation **unverified**)

In [ ]:
# =====================================================================================
# A1 -- "Chai et al. 2024" Transformer, no imbalance handling -- CITATION UNVERIFIED
# =====================================================================================
# I could not locate a paper matching this description under a first-author "Chai"
# despite extensive search (2024 IoT/CICIoT-2023 transformer-IDS literature is large
# and "Chai" did not turn up as a first author on any close match). Rather than
# reproduce architecture details I can't verify against a real source, this implements
# a standard "vanilla" per-feature-token Transformer classifier -- the common pattern
# across the many real 2024 CICIoT-2023 transformer-IDS papers found during search --
# as a placeholder. CONFIRM-ME: if you have the actual DOI/link, tell me and I'll
# rebuild this to match it exactly.
#
# "No imbalance handling" is implemented literally: this is the one baseline that
# receives neither this fold's resampling (Section III-D) nor class-weighted loss --
# see the data-routing block in the main CV loop. Every other model in MODEL_ORDER
# gets both, so A1 is a genuine contrast baseline for what the rest of that machinery
# is worth.

class A1_TransformerNet(nn.Module):
    def __init__(self, in_dim, n_classes, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.token_proj = nn.Linear(1, d_model)
        self.pos_enc = nn.Parameter(torch.zeros(in_dim, d_model))
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
                                                dropout=dropout, activation="relu", batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, n_classes)
        nn.init.trunc_normal_(self.pos_enc, std=0.02)

    def forward(self, x):
        tokens = self.token_proj(x.unsqueeze(-1)) + self.pos_enc.unsqueeze(0)  # (B, F, d_model)
        enc = self.encoder(tokens)
        pooled = enc.mean(dim=1)     # plain mean pool -- no [CLS]/attention pooling (cf. CAFE-TF)
        return self.head(pooled)


def build_a1_transformer(n_features, n_classes, cfg=A1_TRANSFORMER_CONFIG):
    model_fn = lambda d, c: A1_TransformerNet(d, c, d_model=cfg["d_model"], n_heads=cfg["n_heads"],
                                               n_layers=cfg["n_layers"], d_ff=cfg["d_ff"], dropout=cfg["dropout"])
    return TorchClassifierWrapper(model_fn, epochs=cfg["epochs"], lr=cfg["lr"], batch_size=cfg["batch_size"])

print("A1 (Transformer, unverified citation -- see comment above) defined.")


## A2: Sahin & Uludag (2023) -- Compact DNN on Interpacket Arrival Time (IAT) Alone

In [ ]:
# =====================================================================================
# A2 -- Sahin & Uludag (2023), "Less is More": compact DNN on interpacket arrival
# time (IAT) alone. VERIFIED against the full preprint (Research Square,
# doi:10.21203/rs.3.rs-3398752/v1). Their Table 5 feature list (46 features + label)
# is the same CICIoT-2023 export this pipeline uses, and Random Forest feature
# importance on that dataset put IAT alone ahead of every other single feature and
# even a 25-feature subset for their DNN's F1 -- that's the entire point of the paper.
# =====================================================================================

class A2_SahinUludagNet(nn.Module):
    """Reproduces their reported Keras Sequential exactly: Dense(256, relu) ->
    Dropout(0.5) -> Dense(128, relu) -> Dense(64, sigmoid) -> Dense(n_classes).
    in_dim is 1 here (IAT only), not this pipeline's usual 26."""
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.Sigmoid(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)   # raw logits; softmax applied by cross_entropy / predict_proba


def build_a2_iat_dnn(cfg=A2_IAT_DNN_CONFIG):
    return TorchClassifierWrapper(lambda d, c: A2_SahinUludagNet(d, c),
                                   epochs=cfg["epochs"], lr=cfg["lr"], batch_size=cfg["batch_size"])

print("A2 (Sahin & Uludag, IAT-only DNN) defined.")


## A4: GIDS -- GraphSAGE + LSTM Neighbour Aggregator (citation **uncertain**: verified as Yang et al. 2025, not "Zhu et al.")

In [ ]:
# =====================================================================================
# A4 -- GraphSAGE with an LSTM neighbour aggregator ("GIDS")
# Citation given as "Zhu et al., Symmetry" -- NOT VERIFIED under that author. Closest
# verified match on topic + venue + year: Yang, S.; Pan, W.; Li, M.; Yin, M.; Ren, H.;
# Chang, Y.; Liu, Y.; Zhang, S.; Lou, F. (2025) "Industrial Internet of Things Intrusion
# Detection System Based on Graph Neural Network." Symmetry, 17(7), 997,
# doi:10.3390/sym17070997. Implemented against THIS paper. CONFIRM-ME if you meant a
# different "Zhu et al." paper.
#
# ADAPTATION: the paper builds its graph from device/IP identity fields (device-level
# + gateway-level traffic graphs) that don't exist in this pipeline's 26 anonymised
# flow-statistic features. Each mini-batch is turned into a k-nearest-neighbour graph
# in feature space instead -- the standard fallback for GNNs on purely tabular data.
# The LSTM aggregation and update rule (their Eq. 4:
#   H_v^l = ReLU(W^l . Concat(H_v^{l-1}, LSTM_{u in N(v)} H_u^{l-1})))
# is otherwise implemented as specified.
# =====================================================================================

class LSTMNeighborAggregator(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.lstm = nn.LSTM(input_size=d_model, hidden_size=d_model, batch_first=True)
        self.update = nn.Linear(2 * d_model, d_model)

    def forward(self, H, knn_idx):
        neighbor_feats = H[knn_idx]                # (B, k, d_model)
        _, (h_n, _) = self.lstm(neighbor_feats)
        agg = h_n.squeeze(0)                        # (B, d_model)
        return F.relu(self.update(torch.cat([H, agg], dim=1)))


def batch_knn_indices(X, k):
    """Euclidean k-NN within a mini-batch (self excluded) -- see adaptation note above."""
    dist = torch.cdist(X, X)
    dist.fill_diagonal_(float("inf"))
    k_eff = min(k, X.shape[0] - 1)
    _, idx = torch.topk(dist, k_eff, dim=1, largest=False)
    return idx


class A4_GIDSNet(nn.Module):
    def __init__(self, in_dim, n_classes, d_model, n_layers, k_neighbors):
        super().__init__()
        self.in_proj = nn.Linear(in_dim, d_model)
        self.layers = nn.ModuleList([LSTMNeighborAggregator(d_model) for _ in range(n_layers)])
        self.k = k_neighbors
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        H = self.in_proj(x)
        knn_idx = batch_knn_indices(x.detach(), self.k)   # graph rebuilt fresh each batch
        for layer in self.layers:
            H = layer(H, knn_idx)
        return self.head(H)


def build_a4_gids(n_features, n_classes, cfg=A4_GIDS_CONFIG):
    model_fn = lambda d, c: A4_GIDSNet(d, c, d_model=cfg["d_model"], n_layers=cfg["n_layers"],
                                        k_neighbors=cfg["k_neighbors"])
    return TorchClassifierWrapper(model_fn, epochs=cfg["epochs"], lr=cfg["lr"], batch_size=cfg["batch_size"])

print("A4 (GIDS: GraphSAGE + LSTM aggregator -- citation unverified, see comment above) defined.")


## A5: GCN-DQN -- GCN + Multi-Head Attention + DQN-Adaptive Attention Weighting

In [ ]:
import collections

# =====================================================================================
# A5 -- GCN-DQN: GCN + multi-head self-attention + DQN-adaptive attention weighting
# Mwiga, K.; Dida, M.; Maglaras, L.; Mohsin, A.; Janicke, H.; Sarker, I.H. (2026).
# "Graph Convolution Neural Network and Deep Q-Network Optimization-Based Intrusion
# Detection with Explainability Analysis." Sensors, 26(5), 1421, doi:10.3390/s26051421.
# VERIFIED -- full methodology (their Eq. 1-19) read from the paper.
#
# ADAPTATION NOTES (read before trusting these numbers against the paper):
#  1. Graph construction (their Eq. 1/3): the paper thresholds a full-dataset
#     Euclidean-distance adjacency matrix, which is O(n^2) and infeasible at this
#     pipeline's scale. Rebuilt per mini-batch instead -- same rule, bounded cost.
#     They never state their distance threshold, so a per-batch distance PERCENTILE
#     (GCN_DQN edge_percentile) is used instead of a hard-coded constant, which stays
#     well-behaved across folds/scalers where a fixed number would not.
#  2. DQN state (their Eq. 8) mixes a matrix A_t with two scalars and a loss value
#     into one fixed-size vector without saying how. Implemented as a single scalar
#     "attention gain" (initialised to 1.0, multiplying attention scores) that the DQN
#     nudges up/down/reset/no-op by a fixed step (their Section 3.6.2, alpha=0.05) --
#     the only reading consistent with a 4-dim state and one additive step size. This
#     is the biggest interpretive gap versus the paper.
#  3. GCN depth is not stated explicitly; 2 layers (the common default) is used.
# =====================================================================================

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, M):
        return F.relu(M @ self.W(H))


def batch_normalized_adjacency(X, edge_percentile):
    """Their Eq. 1 (Euclidean threshold) + Eq. 3 (symmetric-normalised, self-loops)."""
    B = X.shape[0]
    if B <= 1:
        return torch.eye(max(B, 1), device=X.device)[:B, :B]
    dist = torch.cdist(X, X)
    off_diag = dist[~torch.eye(B, dtype=torch.bool, device=X.device)]
    thresh = torch.quantile(off_diag, edge_percentile)
    A = (dist < thresh).float()
    A.fill_diagonal_(0.0)
    A_tilde = A + torch.eye(B, device=X.device)
    deg = A_tilde.sum(dim=1)
    d_inv_sqrt = torch.pow(deg.clamp(min=1e-8), -0.5)
    return d_inv_sqrt.unsqueeze(1) * A_tilde * d_inv_sqrt.unsqueeze(0)


class A5_GCNAttentionNet(nn.Module):
    def __init__(self, in_dim, n_classes, d_model, n_gcn_layers, n_heads, edge_percentile):
        super().__init__()
        self.in_proj = nn.Linear(in_dim, d_model)
        self.gcn_layers = nn.ModuleList([GCNLayer(d_model, d_model) for _ in range(n_gcn_layers)])
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)
        self.head = nn.Linear(d_model, n_classes)
        self.edge_percentile = edge_percentile
        self.register_buffer("attn_gain", torch.tensor(1.0))   # DQN-controlled scalar, see note 2

    def forward(self, x, return_embed=False):
        M = batch_normalized_adjacency(x.detach(), self.edge_percentile)
        H = self.in_proj(x)
        for layer in self.gcn_layers:
            H = layer(H, M)
        seq = (H * self.attn_gain).unsqueeze(1)
        attn_out, _ = self.attn(seq, seq, seq)
        logits = self.head(attn_out.squeeze(1))
        return (logits, H) if return_embed else logits


class TinyDQN(nn.Module):
    """State = [attn_gain, mean(H), std(H), loss] -> Q-value per action."""
    def __init__(self, state_dim=4, n_actions=4, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_actions))

    def forward(self, s):
        return self.net(s)


class A5_GCNDQNWrapper:
    """Custom wrapper (not TorchClassifierWrapper): supervised GCN+attention updates
    are interleaved with periodic DQN steps that adjust attn_gain (their Section 3.6)."""
    ACTIONS = [+0.05, -0.05, "reset", "noop"]

    def __init__(self, cfg, n_classes=N_CLASSES):
        self.cfg = cfg
        self.n_classes = n_classes
        self.model = None

    def fit(self, X, y, class_weight=None):
        cfg = self.cfg
        self.model = A5_GCNAttentionNet(X.shape[1], self.n_classes, d_model=cfg["d_model"],
                                         n_gcn_layers=cfg["n_gcn_layers"], n_heads=cfg["n_heads"],
                                         edge_percentile=cfg["edge_percentile"]).to(DEVICE)
        dqn = TinyDQN().to(DEVICE)
        dqn_target = TinyDQN().to(DEVICE)
        dqn_target.load_state_dict(dqn.state_dict())

        w = torch.tensor(class_weight, dtype=torch.float32).to(DEVICE) if class_weight is not None else None
        opt = torch.optim.Adam(list(self.model.parameters()) + list(dqn.parameters()),
                                lr=cfg["lr"], weight_decay=cfg["weight_decay"])

        Xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.long)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=cfg["batch_size"], shuffle=True)

        epsilon = cfg["epsilon"]
        loss_threshold, prev_loss = None, None    # threshold = first batch's loss (their Sec 3.6)
        replay = collections.deque(maxlen=2000)
        self.loss_history = []

        self.model.train()
        for epoch in range(cfg["epochs"]):
            epoch_loss, n_batches = 0.0, 0
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits, H = self.model(xb, return_embed=True)
                loss = F.cross_entropy(logits, yb, weight=w)
                opt.zero_grad(); loss.backward(); opt.step()
                loss_val = loss.item()
                if loss_threshold is None:
                    loss_threshold = loss_val

                if loss_val > loss_threshold:   # DQN attention-gain adjustment (Sec 3.6)
                    state = torch.tensor([self.model.attn_gain.item(), H.mean().item(),
                                           H.std().item(), loss_val], dtype=torch.float32).to(DEVICE)
                    if random.random() < epsilon:
                        action = random.randrange(len(self.ACTIONS))
                    else:
                        with torch.no_grad():
                            action = int(dqn(state).argmax().item())

                    old_gain, act = self.model.attn_gain.item(), self.ACTIONS[action]
                    new_gain = 1.0 if act == "reset" else (old_gain if act == "noop" else old_gain + act)
                    self.model.attn_gain.fill_(new_gain)

                    reward = (prev_loss - loss_val) if prev_loss is not None else 0.0
                    with torch.no_grad():
                        logits2, H2 = self.model(xb, return_embed=True)
                        next_loss = F.cross_entropy(logits2, yb, weight=w).item()
                    next_state = torch.tensor([new_gain, H2.mean().item(), H2.std().item(), next_loss],
                                               dtype=torch.float32).to(DEVICE)
                    replay.append((state, action, reward, next_state))

                    if len(replay) >= cfg["dqn_batch_size"]:
                        batch = random.sample(replay, cfg["dqn_batch_size"])
                        s_b = torch.stack([b[0] for b in batch])
                        a_b = torch.tensor([b[1] for b in batch], device=DEVICE)
                        r_b = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=DEVICE)
                        s2_b = torch.stack([b[3] for b in batch])
                        with torch.no_grad():
                            target_q = r_b + cfg["gamma"] * dqn_target(s2_b).max(dim=1).values
                        q_pred = dqn(s_b).gather(1, a_b.unsqueeze(1)).squeeze(1)
                        dqn_loss = F.mse_loss(q_pred, target_q)
                        opt.zero_grad(); dqn_loss.backward(); opt.step()

                    epsilon = max(cfg["epsilon_min"], epsilon * cfg["epsilon_decay"])
                    prev_loss = loss_val

                epoch_loss += loss_val; n_batches += 1

            if (epoch + 1) % cfg["target_sync_every"] == 0:
                dqn_target.load_state_dict(dqn.state_dict())
            self.loss_history.append(epoch_loss / max(1, n_batches))
        return self

    @torch.no_grad()
    def predict_proba(self, X):
        self.model.eval()
        Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        probs = []
        for i in range(0, len(Xt), self.cfg["batch_size"]):
            logits = self.model(Xt[i:i + self.cfg["batch_size"]])
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        return np.concatenate(probs, axis=0)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def build_a5_gcn_dqn(cfg=A5_GCN_DQN_CONFIG, n_classes=N_CLASSES):
    return A5_GCNDQNWrapper(cfg, n_classes=n_classes)

print("A5 (GCN-DQN) defined.")


## A3: QLoRA-LLaMA-3.2-1B + RAG (**expensive** -- read the cell's notes before setting `RUN_A3_LLM_RAG = True`)

In [ ]:
import re

# =====================================================================================
# A3 -- QLoRA-tuned LLaMA-3.2-1B + RAG
# Sudasinghe, P.B.; Liyanage, K.S.K.; Pussewalage, H.S.G. (2026). "Lightweight LLMs
# for Network Attack Detection in IoT Networks." Presented at ComComAp 2025 (Madrid,
# 14-17 Dec 2025); arXiv:2601.15269. VERIFIED against the full arXiv text -- structured-
# to-text conversion, QLoRA config (r=16, alpha=32, dropout=0.1, NF4 4-bit), and RAG
# design (cosine-similarity retrieval, top-20 candidates -> top-3 in context) all
# confirmed from the source.
#
# READ THIS BEFORE ENABLING RUN_A3_LLM_RAG:
#  - Scale: the paper trains/tests on 500/100 samples per class (24 classes), not this
#    pipeline's full ~278k-row resampled fold. Fine-tuning an LLM on the full fold
#    would take vastly longer than every other baseline in this notebook COMBINED.
#    A3_LLM_RAG_CONFIG defaults to a bounded stratified per-class subsample instead,
#    mirroring the paper's own scale -- raise max_train_per_class/max_eval_per_class
#    if you have the compute budget for a closer-to-full-data run.
#  - Label space: fine-tuned on THIS pipeline's 8-class grouping, not the paper's 24
#    raw classes -- consistent with every other model in this benchmark, but a real
#    change from the source paper's task.
#  - RAG semantics: the paper's knowledge base is built from attack classes held OUT
#    of training entirely (true zero-shot generalisation to unseen attacks). This
#    pipeline trains on all 8 classes every fold, so there's no "unseen class" set to
#    build one from. Retrieval is repurposed here as similarity-based in-context
#    example selection from the fold's own training subsample -- it demonstrates a
#    different thing than the paper's RAG experiment. Set use_rag=False in
#    A3_LLM_RAG_CONFIG to isolate plain QLoRA fine-tuning without this change.
#  - predict_proba: the paper evaluates via greedy generation + string matching, which
#    doesn't give a probability vector. This scores each of the 8 class-name strings'
#    log-likelihood under the model and softmaxes across them instead -- a standard
#    "LLM as classifier" technique, needed so this baseline can be evaluated with the
#    same F1/AUC/Friedman machinery as everything else in this notebook.
#  - THIS CODE IS UNTESTED END-TO-END. It follows current transformers/peft/
#    bitsandbytes APIs and the text-conversion/RAG/scoring logic was verified against
#    a local toy model, but the actual LLaMA-3.2-1B load + QLoRA fine-tune has not run
#    anywhere with real weights (no GPU / no Hugging Face Hub access in the environment
#    that wrote this). Smoke-test on a small subsample before trusting a full run.
# =====================================================================================

def humanize_feature_name(col_name):
    """'psh_flag_number' -> 'Psh Flag Number'. Short all-caps tokens (IAT, TCP, UDP,
    HTTP, ...) are kept as acronyms rather than title-cased. NOTE: the paper also does
    a few fully semantic renames (IAT -> 'Time Between Packets', rst_count -> 'Packets
    with RST Flag') this generic rule can't reproduce without a per-feature lookup the
    paper doesn't fully enumerate -- an approximation, not a verified match."""
    words = re.sub(r"[_\-]+", " ", col_name).split()
    return " ".join(w if (w.isupper() and len(w) <= 5) else w.capitalize() for w in words)


def row_to_prompt(row, feature_names, class_names, n_decimals=6):
    feat_str = "; ".join(f"{humanize_feature_name(c)} = {row[c]:.{n_decimals}f}" for c in feature_names)
    return (f"Task: Network Attack Classification\n"
            f"Input Features: {{{feat_str}}}\n"
            f"Possible Classes: [{', '.join(class_names)}]\n"
            f"Answer:")


def build_retrieval_index(X_raw_df):
    X = X_raw_df.values.astype(np.float32)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms


def retrieve_top_examples(query_row_raw, index_normed, index_labels, feature_names, top_k, top_n):
    q = query_row_raw[feature_names].values.astype(np.float32)
    qn = q / (np.linalg.norm(q) or 1.0)
    sims = index_normed @ qn
    k = min(top_k, len(sims))
    top_k_idx = np.argpartition(-sims, k - 1)[:k]
    top_k_idx = top_k_idx[np.argsort(-sims[top_k_idx])]
    chosen = top_k_idx[:top_n]
    return chosen, index_labels[chosen]


def rag_prompt(query_row, feature_names, class_names, retrieved_rows, retrieved_labels, n_decimals=6):
    examples = []
    for r, lab in zip(retrieved_rows, retrieved_labels):
        ex_feats = "; ".join(f"{humanize_feature_name(c)}={r[c]:.{n_decimals}f}" for c in feature_names)
        examples.append(f"Input: {{{ex_feats}}}\nAnswer: {class_names[lab]}")
    return f"Retrieved Examples:\n" + "\n".join(examples) + f"\n\n{row_to_prompt(query_row, feature_names, class_names, n_decimals)}"


@torch.no_grad()
def score_class_strings(model, tokenizer, prompt, class_names, device, max_seq_len):
    """Teacher-forced log-likelihood of each class string right after the prompt,
    softmaxed across classes -- see 'predict_proba' note above."""
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=max_seq_len).input_ids.to(device)
    scores = []
    for cname in class_names:
        cont_ids = tokenizer(" " + cname, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
        full_ids = torch.cat([prompt_ids, cont_ids], dim=1)
        logits = model(full_ids).logits[:, :-1, :]
        targets = full_ids[:, 1:]
        logprobs = F.log_softmax(logits, dim=-1)
        cont_len = cont_ids.shape[1]
        tok_logprobs = logprobs[0, -cont_len:, :].gather(1, targets[0, -cont_len:].unsqueeze(1)).squeeze(1)
        scores.append(tok_logprobs.sum().item())
    return F.softmax(torch.tensor(scores), dim=0).numpy()


def stratified_subsample(X_df, y, max_per_class, seed=SEED):
    rng = np.random.RandomState(seed)
    keep_idx = []
    for c in np.unique(y):
        c_idx = np.where(y == c)[0]
        rng.shuffle(c_idx)
        keep_idx.extend(c_idx[:max_per_class].tolist())
    keep_idx = np.array(sorted(keep_idx))
    return X_df.iloc[keep_idx].reset_index(drop=True), y[keep_idx]


class A3_LLMRAGWrapper:
    def __init__(self, cfg, n_classes=N_CLASSES):
        self.cfg = cfg
        self.n_classes = n_classes
        self.model = self.tokenizer = None
        self.retrieval_index = self.retrieval_labels = self.retrieval_rows = None
        self.feature_names = None

    def fit(self, X_train_raw_df, y_train, class_weight=None):
        # class_weight is accepted for interface consistency but unused: the paper's
        # loss is masked cross-entropy over generated tokens, not this pipeline's
        # per-class weighting scheme (analogous to the anomaly detectors ignoring it).
        cfg = self.cfg
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, TaskType

        self.feature_names = list(X_train_raw_df.columns)
        Xsub, ysub = stratified_subsample(X_train_raw_df, y_train, cfg["max_train_per_class"])
        print(f"    A3: fine-tuning on {len(Xsub)} rows ({cfg['max_train_per_class']}/class cap)")

        self.tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token   # decoder-only models lack one

        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_use_double_quant=True)
        base_model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], quantization_config=bnb_config,
                                                           device_map="auto")
        lora_config = LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=cfg["lora_dropout"],
                                  bias="none", task_type=TaskType.CAUSAL_LM, target_modules=cfg["target_modules"])
        self.model = get_peft_model(base_model, lora_config)
        self.model.print_trainable_parameters()

        # Build (prompt, target) pairs and mask the loss to the "Answer: <class>" span
        # only (paper's custom data collator; here as explicit label masking).
        input_ids_list, labels_list = [], []
        for i in range(len(Xsub)):
            prompt = row_to_prompt(Xsub.iloc[i], self.feature_names, CLASS_ORDER)
            target = " " + CLASS_ORDER[ysub[i]] + self.tokenizer.eos_token
            p_ids = self.tokenizer(prompt, truncation=True, max_length=cfg["max_seq_len"]).input_ids
            t_ids = self.tokenizer(target, add_special_tokens=False).input_ids
            ids = p_ids + t_ids
            labels = [-100] * len(p_ids) + t_ids
            input_ids_list.append(ids); labels_list.append(labels)

        opt = torch.optim.AdamW(self.model.parameters(), lr=cfg["lr"])
        self.model.train()
        self.loss_history = []
        n = len(input_ids_list)
        for epoch in range(cfg["epochs"]):
            perm = np.random.permutation(n)
            epoch_loss, n_batches = 0.0, 0
            for start in range(0, n, cfg["batch_size"]):
                batch_idx = perm[start:start + cfg["batch_size"]]
                max_len = max(len(input_ids_list[j]) for j in batch_idx)
                pad_id = self.tokenizer.pad_token_id
                ids_b, labels_b, mask_b = [], [], []
                for j in batch_idx:
                    ids, labels = input_ids_list[j], labels_list[j]
                    pad = max_len - len(ids)
                    ids_b.append(ids + [pad_id] * pad)
                    labels_b.append(labels + [-100] * pad)
                    mask_b.append([1] * len(ids) + [0] * pad)
                ids_t = torch.tensor(ids_b).to(self.model.device)
                labels_t = torch.tensor(labels_b).to(self.model.device)
                mask_t = torch.tensor(mask_b).to(self.model.device)

                out = self.model(input_ids=ids_t, attention_mask=mask_t, labels=labels_t)
                opt.zero_grad(); out.loss.backward(); opt.step()
                epoch_loss += out.loss.item(); n_batches += 1
            self.loss_history.append(epoch_loss / max(1, n_batches))
            print(f"    A3 epoch {epoch + 1}/{cfg['epochs']}: loss={self.loss_history[-1]:.4f}")

        # Retrieval knowledge base = this same fine-tuning subsample (see RAG note above)
        self.retrieval_rows = Xsub
        self.retrieval_labels = ysub
        self.retrieval_index = build_retrieval_index(Xsub)
        return self

    def predict_proba(self, X_test_raw_df):
        cfg = self.cfg
        self.model.eval()
        device = self.model.device
        n = len(X_test_raw_df)
        proba = np.zeros((n, self.n_classes))
        for i in range(n):
            row = X_test_raw_df.iloc[i]
            if cfg["use_rag"]:
                idx, labels = retrieve_top_examples(row, self.retrieval_index, self.retrieval_labels,
                                                     self.feature_names, cfg["rag_top_k_candidates"],
                                                     cfg["rag_top_n_context"])
                prompt = rag_prompt(row, self.feature_names, CLASS_ORDER,
                                     [self.retrieval_rows.iloc[j] for j in idx], labels)
            else:
                prompt = row_to_prompt(row, self.feature_names, CLASS_ORDER)
            proba[i] = score_class_strings(self.model, self.tokenizer, prompt, CLASS_ORDER,
                                            device, cfg["max_seq_len"])
        return proba

    def predict(self, X_test_raw_df):
        return self.predict_proba(X_test_raw_df).argmax(axis=1)


def build_a3_llm_rag(cfg=A3_LLM_RAG_CONFIG, n_classes=N_CLASSES):
    return A3_LLMRAGWrapper(cfg, n_classes=n_classes)


def fit_predict_a3(model, X_train_raw, y_train, X_test_raw, y_test, cfg=A3_LLM_RAG_CONFIG):
    """Called from the main CV loop instead of the standard model.fit/predict_proba
    pair, because A3 needs raw (unscaled, named) features and evaluates on a bounded
    subsample rather than the full fold -- see the notes at the top of this cell."""
    model.fit(X_train_raw, y_train)
    X_eval, y_eval = stratified_subsample(X_test_raw, y_test, cfg["max_eval_per_class"])
    print(f"    A3: evaluating on {len(X_eval)} rows ({cfg['max_eval_per_class']}/class cap)")
    proba = model.predict_proba(X_eval)
    return proba, y_eval

print("A3 (QLoRA-LLaMA + RAG -- read the notes above before enabling) defined.")


## Novel Model 1: PANE

In [ ]:
# =====================================================================================
# CELL 9 -- Novel Model 1: PANE (Phase-Adaptive Neural Estimator)
# =====================================================================================
# phi(t) = exp(i*2*pi*f_k*t), f_k in {f_1,...,f_K} learned (Section V-A). Flow-level
# tabular data has no explicit time index, so each feature's own scaled value is used
# as the phase-modulation variable t (a standard way to lift phase-based ideas from
# packet-sequence data onto flow-level features) -- this is a design choice worth
# confirming against your original implementation if the numbers look off.
# gru_hidden is CONFIRM-ME (Cell 2) since the manuscript never states it explicitly.

class PANE(nn.Module):
    def __init__(self, in_dim, n_classes, gru_hidden=128, gru_layers=4, n_frequencies=16):
        super().__init__()
        self.freqs = nn.Parameter(torch.rand(n_frequencies) * 5.0)  # learned f_k
        self.phase_proj = nn.Linear(2 * n_frequencies, 32)
        self.gru = nn.GRU(input_size=in_dim + 32, hidden_size=gru_hidden,
                           num_layers=gru_layers, batch_first=True,
                           bidirectional=True, dropout=0.2 if gru_layers > 1 else 0)
        self.attn_pool = nn.Linear(gru_hidden * 2, 1)
        self.head = nn.Linear(gru_hidden * 2, n_classes)

    def forward(self, x):
        # x: (B, F). Build a per-sample phase signature from the mean feature value.
        t = x.mean(dim=1, keepdim=True)                    # (B, 1)
        angle = 2 * torch.pi * self.freqs.unsqueeze(0) * t  # (B, K)
        phase = torch.cat([torch.cos(angle), torch.sin(angle)], dim=1)  # (B, 2K) -- Re/Im of exp(i*2*pi*f*t)
        phase = self.phase_proj(phase)                      # (B, 32)
        seq_in = torch.cat([x, phase], dim=1).unsqueeze(1)  # (B, 1, F+32)
        out, _ = self.gru(seq_in)                           # (B, 1, 2*hidden)
        weights = torch.softmax(self.attn_pool(out), dim=1)
        pooled = (out * weights).sum(dim=1)
        return self.head(pooled)


def build_pane(n_features, n_classes, cfg=PANE_CONFIG):
    model_fn = lambda d, c: PANE(d, c, gru_hidden=cfg["gru_hidden"],
                                  gru_layers=cfg["gru_layers"], n_frequencies=cfg["n_frequencies"])
    return TorchClassifierWrapper(model_fn, epochs=cfg["epochs"], lr=cfg["lr"])

print("PANE defined.")


## Novel Model 2: CAFE-TF

In [ ]:
# =====================================================================================
# CELL 10 -- Novel Model 2: CAFE-TF (Cross-Attention Feature Ensemble Transformer)
# =====================================================================================
# Architecture matches Section V-B exactly: per-group linear projection -> learnable
# positional encoding -> [CLS] token -> 4 pre-LN transformer encoder blocks (8 heads,
# d_ff=512) -> 2-layer GELU classification head. Same structure used earlier this
# session to verify the manuscript's parameter count.

class CAFE_TF(nn.Module):
    def __init__(self, feature_groups, n_classes, d_model=128, n_heads=8, n_layers=4, d_ff=512):
        super().__init__()
        self.feature_groups = feature_groups
        self.proj = nn.ModuleList([nn.Linear(d, d_model) for d in feature_groups])
        self.pos_enc = nn.Parameter(torch.zeros(len(feature_groups), d_model))
        self.cls_token = nn.Parameter(torch.zeros(1, d_model))
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                                dim_feedforward=d_ff, activation="gelu",
                                                norm_first=True, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_ln = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Linear(64, n_classes))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        # x: (B, sum(feature_groups))
        splits = torch.split(x, self.feature_groups, dim=1)
        tokens = [proj(s) for proj, s in zip(self.proj, splits)]     # G x (B, d_model)
        tokens = torch.stack(tokens, dim=1) + self.pos_enc.unsqueeze(0)  # (B, G, d_model)
        cls = self.cls_token.unsqueeze(0).expand(x.size(0), -1, -1)
        seq = torch.cat([cls, tokens], dim=1)                        # (B, G+1, d_model)
        enc = self.encoder(seq)
        enc = self.final_ln(enc)
        return self.head(enc[:, 0, :])  # [CLS] representation


def build_cafe_tf(n_features, n_classes, group_sizes=None, cfg=CAFE_TF_CONFIG):
    # v9: group_sizes is now supplied per-fold by run_full_evaluation (from
    # reorder_columns_by_group), since the filter is fit per fold and the retained
    # feature set -- and therefore each group's true size -- can vary slightly fold
    # to fold. Falls back to the static config only if called standalone.
    groups = group_sizes if group_sizes is not None else cfg["feature_groups"]
    assert sum(groups) == n_features, (
        f"CAFE-TF's group sizes {groups} sum to {sum(groups)} but the data has "
        f"{n_features} features."
    )
    model_fn = lambda d, c: CAFE_TF(groups, c, d_model=cfg["d_model"], n_heads=cfg["n_heads"],
                                     n_layers=cfg["n_layers"], d_ff=cfg["d_ff"])
    return TorchClassifierWrapper(model_fn, epochs=cfg["epochs"], lr=cfg["lr"])

print("CAFE-TF defined.")


## Novel Model 3: ARID (class-weighted adversarial training)

In [ ]:
# =====================================================================================
# CELL 11 -- Novel Model 3: ARID (v2: class-weighted PGD adversarial training)
# =====================================================================================

class ARIDNet(nn.Module):
    def __init__(self, in_dim, n_classes, hidden_dims):
        super().__init__()
        widths = [in_dim] + hidden_dims
        self.blocks = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.proj = nn.ModuleList()
        for i in range(len(widths) - 1):
            self.blocks.append(nn.Linear(widths[i], widths[i + 1]))
            self.bns.append(nn.BatchNorm1d(widths[i + 1]))
            self.proj.append(nn.Linear(widths[i], widths[i + 1]) if widths[i] != widths[i + 1] else nn.Identity())
        self.out = nn.Linear(widths[-1], n_classes)

    def forward(self, x):
        h = x
        for lin, bn, proj in zip(self.blocks, self.bns, self.proj):
            residual = proj(h)
            h = F.relu(bn(lin(h)) + residual)
        return self.out(h)


def pgd_attack(model, x, y, epsilon, alpha, steps):
    x_adv = x.clone().detach() + torch.empty_like(x).uniform_(-epsilon, epsilon)
    for _ in range(steps):
        x_adv.requires_grad_(True)
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + alpha * grad.sign()
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon).detach()
    return x_adv


class ARIDWrapper(TorchClassifierWrapper):
    def __init__(self, hidden_dims, epochs, lr, pgd_epsilon, pgd_alpha, pgd_steps, adv_lambda, n_classes=N_CLASSES):
        super().__init__(lambda d, c: ARIDNet(d, c, hidden_dims), epochs=epochs, lr=lr, n_classes=n_classes)
        self.pgd_epsilon, self.pgd_alpha, self.pgd_steps = pgd_epsilon, pgd_alpha, pgd_steps
        self.adv_lambda = adv_lambda

    def fit(self, X, y, class_weight=None):
        n_features = X.shape[1]
        self.model = self.model_fn(n_features, self.n_classes).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)

        w = None
        if class_weight is not None:
            w = torch.tensor(class_weight, dtype=torch.float32).to(DEVICE)

        Xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.long)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=self.batch_size, shuffle=True)

        for epoch in range(self.epochs):
            self.model.train()
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                x_adv = pgd_attack(self.model, xb, yb, self.pgd_epsilon, self.pgd_alpha, self.pgd_steps)
                opt.zero_grad()
                clean_loss = F.cross_entropy(self.model(xb), yb, weight=w)
                adv_loss = F.cross_entropy(self.model(x_adv), yb, weight=w)
                loss = clean_loss + self.adv_lambda * adv_loss
                loss.backward()
                opt.step()
            sched.step()
        return self


def build_arid(n_features, n_classes, cfg=ARID_CONFIG):
    return ARIDWrapper(cfg["hidden_dims"], cfg["epochs"], cfg["lr"], cfg["pgd_epsilon"],
                        cfg["pgd_alpha"], cfg["pgd_steps"], cfg["adv_lambda"], n_classes=n_classes)

print("ARID defined (v2, class-weighted).")


## Novel Model 4: FedHAN (class-weighted federated training)

In [ ]:
# ---------------------------------------------------------------------------------
# BUGFIX (v6) support: identify BatchNorm buffers so they can be held out of
# federated aggregation, and recalibrate them once on the server afterwards.
# ---------------------------------------------------------------------------------
BN_BUFFER_SUFFIXES = ("running_mean", "running_var", "num_batches_tracked")
FEDHAN_LOCAL_GRAD_CLIP = 5.0
FEDHAN_BASE_LR = 0.01
FEDHAN_LR_DECAY = 0.97      # per communication round; 0.97^96 ~= 0.055


def is_bn_buffer(key):
    return any(key.endswith(s) for s in BN_BUFFER_SUFFIXES)


@torch.no_grad()
def recalibrate_bn(model, X, n_batches=50, batch_size=512):
    """Populate BatchNorm running statistics for the aggregated global model.

    Because BN statistics are never averaged across clients (Section V-D), the global
    model's buffers would otherwise remain at their initialization values, which makes
    centralized evaluation meaningless. This performs forward passes only -- no
    gradients, no weight updates -- over a sample of the training split.
    """
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.reset_running_stats()
            m.momentum = None          # cumulative moving average over these batches
    model.train()
    Xt = torch.tensor(X, dtype=torch.float32)
    n = min(len(Xt), n_batches * batch_size)
    for i in range(0, n, batch_size):
        model(Xt[i:i + batch_size].to(DEVICE))
    model.eval()
    return model


# =====================================================================================
# Novel Model 4: FedHAN (class-weighted federated training)
# =====================================================================================

class FedHANNet(nn.Module):
    def __init__(self, in_dim, n_classes, channels, kernel_size, fc_dims):
        super().__init__()
        # BUGFIX (v6): adaptive pooling + BN-safe shape inference (see Cell 7).
        self.conv = build_conv_stack(channels, in_dim, kernel_size=kernel_size)
        flat_dim = infer_conv_flat_dim(self.conv, in_dim)
        fc = []
        prev = flat_dim
        for d in fc_dims:
            fc += [nn.Linear(prev, d), nn.ReLU()]
            prev = d
        fc.append(nn.Linear(prev, n_classes))
        self.fc = nn.Sequential(*fc)

    def forward(self, x):
        x = self.conv(x.unsqueeze(1))
        return self.fc(x.flatten(1))


def dirichlet_partition(y, n_clients, alpha, seed=SEED):
    rng = np.random.RandomState(seed)
    n_classes = y.max() + 1
    client_idx = [[] for _ in range(n_clients)]
    for c in range(n_classes):
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(alpha=[alpha] * n_clients)
        splits = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        for client_id, part in enumerate(np.split(idx_c, splits)):
            client_idx[client_id].extend(part.tolist())
    return [np.array(idx) for idx in client_idx]


def client_kl_weight(y_client, n_classes, beta):
    counts = np.bincount(y_client, minlength=n_classes).astype(float)
    q = counts / counts.sum()
    q = np.clip(q, 1e-9, None)
    uniform = 1.0 / n_classes
    kl = np.sum(q * np.log(q / uniform))
    return np.exp(-beta * kl)


def clip_and_noise_state_dict(state_dict, clip_norm, sigma, enabled):
    if not enabled:
        return state_dict
    flat = torch.cat([v.flatten() for v in state_dict.values()])
    total_norm = flat.norm(2)
    scale = min(1.0, clip_norm / (total_norm.item() + 1e-9))
    noised = {}
    for k, v in state_dict.items():
        v_clipped = v * scale
        noise = torch.randn_like(v_clipped) * sigma * clip_norm
        noised[k] = v_clipped + noise
    return noised


class FedHANWrapper:
    def __init__(self, cfg, n_classes=N_CLASSES):
        self.cfg = cfg
        self.n_classes = n_classes
        self.global_model = None

    def fit(self, X, y, class_weight=None):
        n_features = X.shape[1]
        cfg = self.cfg
        w = None
        if class_weight is not None:
            w = torch.tensor(class_weight, dtype=torch.float32).to(DEVICE)

        self.global_model = FedHANNet(n_features, self.n_classes, cfg["cnn_channels"],
                                       cfg["kernel_size"], cfg["fc_dims"]).to(DEVICE)
        client_indices = dirichlet_partition(y, cfg["n_clients"], cfg["dirichlet_alpha"])
        client_weights = [client_kl_weight(y[idx], self.n_classes, cfg["beta_sharpness"])
                           for idx in client_indices]
        w_sum = sum(client_weights)
        client_weights = [wi / w_sum for wi in client_weights]

        Xt_full = torch.tensor(X, dtype=torch.float32)
        yt_full = torch.tensor(y, dtype=torch.long)

        # v9: pre-allocate one local model PER CLIENT, once, instead of the original
        # code's `FedHANNet(...)` call inside the round loop -- which reconstructed a
        # fresh model object 960 times (96 rounds x 10 clients) per fold just to
        # immediately overwrite it via load_state_dict() anyway. This changes NOTHING
        # about the computation: load_state_dict() fully overwrites every parameter
        # regardless of whether the model object is freshly constructed or reused, so
        # the forward/backward pass that follows is identical either way -- this is a
        # guarantee from how nn.Module/state_dict work, not an approximation. Pure
        # allocation-overhead removal, zero effect on results.
        local_models = [FedHANNet(n_features, self.n_classes, cfg["cnn_channels"],
                                   cfg["kernel_size"], cfg["fc_dims"]).to(DEVICE)
                         for _ in client_indices]

        for rnd in range(cfg["comm_rounds"]):
            local_states = []
            for c_id, idx in enumerate(client_indices):
                if len(idx) < 4:
                    local_states.append(None)
                    continue
                local_model = local_models[c_id]
                local_model.load_state_dict(self.global_model.state_dict())
                # BUGFIX (v6): decay the local learning rate across communication
                # rounds. Without it, prolonged federated training destabilises the
                # decision boundary -- measured on this pipeline, FedHAN peaked around
                # round 30 and then degraded, which is what drove the original
                # below-chance result at the configured 96 rounds. AUC stayed high
                # throughout, so the learned representation was never the problem.
                local_lr = FEDHAN_BASE_LR * (FEDHAN_LR_DECAY ** rnd)
                opt = torch.optim.SGD(local_model.parameters(), lr=local_lr, momentum=0.9)
                loader = DataLoader(TensorDataset(Xt_full[idx], yt_full[idx]),
                                     batch_size=128, shuffle=True)
                local_model.train()
                for _ in range(cfg["local_epochs"]):
                    for xb, yb in loader:
                        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                        opt.zero_grad()
                        loss = F.cross_entropy(local_model(xb), yb, weight=w)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(local_model.parameters(),
                                                        max_norm=FEDHAN_LOCAL_GRAD_CLIP)
                        opt.step()
                state = clip_and_noise_state_dict(local_model.state_dict(),
                                                   cfg["dp_clip_norm"], cfg["dp_sigma"],
                                                   cfg["dp_enabled"])
                local_states.append(state)

            valid = [(wi, s) for wi, s in zip(client_weights, local_states) if s is not None]
            w_norm = sum(wi for wi, _ in valid)
            # BUGFIX (v6): aggregate learnable parameters ONLY. BatchNorm running
            # statistics (running_mean / running_var / num_batches_tracked) stay local
            # to each client, matching Section V-D. They are recalibrated once on the
            # server after training (see _recalibrate_bn) so that a single global model
            # can be evaluated centrally in this simulation.
            global_state = self.global_model.state_dict()
            agg_state = {}
            for k in global_state.keys():
                if is_bn_buffer(k):
                    agg_state[k] = global_state[k]          # keep, do not average
                else:
                    agg_state[k] = sum((wi / w_norm) * s[k] for wi, s in valid)
            self.global_model.load_state_dict(agg_state)

            if (rnd + 1) % max(1, cfg["comm_rounds"] // 5) == 0:
                print(f"    FedHAN round {rnd+1}/{cfg['comm_rounds']}  (local lr={local_lr:.5f})")

        # BUGFIX (v6): BN statistics were never aggregated, so populate them once on
        # the server before the global model is used for prediction.
        recalibrate_bn(self.global_model, X)
        return self

    @torch.no_grad()
    def predict_proba(self, X):
        self.global_model.eval()
        Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        probs = []
        for i in range(0, len(Xt), 4096):
            logits = self.global_model(Xt[i:i + 4096])
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        return np.concatenate(probs, axis=0)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def build_fedhan(n_features, n_classes, cfg=FEDHAN_CONFIG):
    return FedHANWrapper(cfg, n_classes=n_classes)

print("FedHAN defined (v2, class-weighted local training).")


## Cascaded Ensemble

In [ ]:
# =====================================================================================
# Cascaded Ensemble
# =====================================================================================
# R(x): screen with PANE (tau1=0.85) -> escalate to CAFE-TF (tau2=0.80) -> escalate to
# a soft vote of ARID+FedHAN. kappa_m(x) = max confidence of model m on x.

class CascadeEnsemble:
    def __init__(self, pane, cafe_tf, arid, fedhan, tau1, tau2, n_classes=N_CLASSES):
        self.pane, self.cafe_tf, self.arid, self.fedhan = pane, cafe_tf, arid, fedhan
        self.tau1, self.tau2 = tau1, tau2
        self.n_classes = n_classes

    def predict_proba(self, X, return_stages=False):
        # v7, changelog D: optional return_stages flag. Default False preserves the
        # exact v6 return signature/behaviour for any existing caller.
        p_pane = self.pane.predict_proba(X)
        p_cafe = self.cafe_tf.predict_proba(X)
        p_arid = self.arid.predict_proba(X)
        p_fedhan = self.fedhan.predict_proba(X)

        kappa_pane = p_pane.max(axis=1)
        kappa_cafe = p_cafe.max(axis=1)

        final_proba = np.zeros((len(X), self.n_classes))
        stage_id = np.zeros(len(X), dtype=int)  # v7: 1/2/3, which stage produced each row

        stage1 = kappa_pane >= self.tau1
        final_proba[stage1] = p_pane[stage1]
        stage_id[stage1] = 1

        remaining = ~stage1
        stage2 = remaining & (kappa_cafe >= self.tau2)
        final_proba[stage2] = p_cafe[stage2]
        stage_id[stage2] = 2

        stage3 = remaining & ~stage2
        # Soft vote: elementwise average of ARID's and FedHAN's probability outputs.
        vote_avg = (p_arid[stage3] + p_fedhan[stage3]) / 2.0
        final_proba[stage3] = vote_avg
        stage_id[stage3] = 3

        if return_stages:
            return final_proba, stage_id
        return final_proba

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def build_cascade(fitted_models, cfg=CASCADE_CONFIG, n_classes=N_CLASSES):
    return CascadeEnsemble(fitted_models["PANE"], fitted_models["CAFE-TF"],
                            fitted_models["ARID"], fitted_models["FedHAN"],
                            tau1=cfg["tau1"], tau2=cfg["tau2"], n_classes=n_classes)

print("Cascade defined.")


## Main 5-Fold Cross-Validation Loop

In [ ]:
# =====================================================================================
# CELL 14 -- Main 5-Fold Cross-Validation Loop (v4: + loss-history persistence for the
# Performance Visualization suite's loss-curve diagnostic plot)
# =====================================================================================
# Order of operations now matches the manuscript exactly (Section III-B/D/E):
#   1. Load data, group 34 raw labels -> 8 classes (Section III-A)
#   2. Fit constant+VIF feature filter ONCE on the full dataset (Section III-B)
#   3. For each fold: scale (fold-internal) -> resample (fold-internal, Section III-D)
#      -> compute sqrt-inverse-frequency class weights on the resampled fold -> train
#      every model with those weights.

RESULTS_CSV = f"{OUTPUT_DIR}/per_fold_results.csv"
PROBA_DIR = f"{OUTPUT_DIR}/fold_probabilities"
LOSS_DIR = f"{OUTPUT_DIR}/loss_history"  # v8: persisted per-model training loss curves
os.makedirs(PROBA_DIR, exist_ok=True)
os.makedirs(LOSS_DIR, exist_ok=True)


def already_done(fold, model_name):
    if not os.path.exists(RESULTS_CSV):
        return False
    done = pd.read_csv(RESULTS_CSV)
    return ((done["fold"] == fold) & (done["model"] == model_name)).any()


def clear_cached_results(model_names, also_clear_proba=True, also_clear_loss=True):
    """v9: This is the fix for the stale-VAE-Detector-row problem documented in
    Section V-A -- already_done() above skips any (fold, model) pair already logged
    in per_fold_results.csv, so a model whose code changed since the last run will
    silently keep reporting its OLD numbers unless its rows are cleared first. Call
    this for any model you've changed before re-running, e.g.:
        clear_cached_results(["VAE Detector"])
    or, to force a full clean re-run of everything:
        clear_cached_results(MODEL_ORDER)
    """
    if isinstance(model_names, str):
        model_names = [model_names]
    if os.path.exists(RESULTS_CSV):
        df = pd.read_csv(RESULTS_CSV)
        before = len(df)
        df = df[~df["model"].isin(model_names)]
        removed = before - len(df)
        df.to_csv(RESULTS_CSV, index=False)
        print(f"Cleared {removed} cached row(s) for {model_names} from {RESULTS_CSV}")
    else:
        print(f"{RESULTS_CSV} does not exist yet -- nothing to clear.")
    if also_clear_proba and os.path.isdir(PROBA_DIR):
        n = 0
        for fn in os.listdir(PROBA_DIR):
            if any(fn.startswith(f"fold") and m.replace(" ", "_") in fn for m in model_names):
                os.remove(os.path.join(PROBA_DIR, fn)); n += 1
        print(f"Cleared {n} cached probability file(s) from {PROBA_DIR}")
    if also_clear_loss and os.path.isdir(LOSS_DIR):
        n = 0
        for fn in os.listdir(LOSS_DIR):
            if any(m.replace(" ", "_") in fn for m in model_names):
                os.remove(os.path.join(LOSS_DIR, fn)); n += 1
        print(f"Cleared {n} cached loss-history file(s) from {LOSS_DIR}")


def append_result(row):
    df_row = pd.DataFrame([row])
    header = not os.path.exists(RESULTS_CSV)
    df_row.to_csv(RESULTS_CSV, mode="a", header=header, index=False)


def run_full_evaluation(data_path=DATA_PATH):
    set_seed(SEED)
    X_raw, y_grouped = load_raw_dataset(data_path)
    X_raw_full = X_raw.copy()   # keep the UNFILTERED raw frame -- A2 needs raw IAT specifically
    y, classes, class_to_idx = encode_labels(y_grouped)
    print(f"Classes ({len(classes)}): {classes}")
    print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))

    # v9: feature filter is now fit PER FOLD inside the loop below (training split
    # only) -- see Cell 4. n_features / FEATURE_NAMES are consequently fold-local.
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_raw, y), start=1):
        print(f"\n===== FOLD {fold}/{N_FOLDS} =====")
        X_train_raw, X_test_raw = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # ---- v9: fold-internal feature filter, then reorder by CAFE-TF's named
        # groups (every other model is column-order-agnostic, so all models share
        # this one consistently-ordered fold dataset) ----
        fold_cols = fit_fold_feature_filter(X_train_raw)
        fold_cols, fold_group_sizes = reorder_columns_by_group(fold_cols)
        X_train_raw = X_train_raw[fold_cols]
        X_test_raw = X_test_raw[fold_cols]
        n_features = len(fold_cols)
        global FEATURE_NAMES
        FEATURE_NAMES = fold_cols   # A3 needs real column names for its text prompts
        print(f"  fold {fold} feature filter: {n_features} features kept, "
              f"CAFE-TF group sizes = {fold_group_sizes}")
        if fold_group_sizes != [4, 1, 5, 4, 4, 2, 4, 2]:
            print(f"  [NOTE] group sizes differ from the original 348k run's "
                  f"[4,1,5,4,4,2,4,2] -- this fold kept a different feature subset. "
                  f"Handled automatically, not an error.")

        # Scale fold-internally (fit on train, apply to test)
        Xtr_scaled, Xte_scaled, scaler = scale_fold(X_train_raw, X_test_raw)

        # Resample fold-internally: DDoS capped to DoS, Recon/BruteForce-Other/Web
        # SMOTE'd aggressively (Section III-D)
        Xtr, ytr = resample_fold(Xtr_scaled, y_train, class_to_idx)
        print(f"  fold {fold}: {len(y_train)} raw train rows -> {len(ytr)} after resampling")
        print("  resampled class counts:", dict(zip(*np.unique(ytr, return_counts=True))))

        # Class weights computed on the RESAMPLED fold (Section III-D: sqrt-inverse-freq)
        class_weight = compute_class_weights(ytr)
        # trivial always-predict-majority accuracy on THIS fold's test split; any model
        # landing under this is broken rather than merely weak (see sanity check below)
        majority_rate = np.bincount(y_test, minlength=N_CLASSES).max() / len(y_test)
        print(f"  majority-class baseline accuracy on test fold: {majority_rate:.4f}")
        print("  class weights:", np.round(class_weight, 3).tolist())

        fitted_novel = {}

        for model_name in MODEL_ORDER:
            if model_name == "Cascaded Ensemble":
                continue
            if already_done(fold, model_name):
                print(f"  [skip, already done] {model_name}")
                continue

            t0 = time.time()
            try:
                if model_name in BASELINE_CONFIG:
                    model = build_baseline_models()[model_name]
                elif model_name == "PANE":
                    model = build_pane(n_features, N_CLASSES)
                elif model_name == "CAFE-TF":
                    model = build_cafe_tf(n_features, N_CLASSES, group_sizes=fold_group_sizes)
                elif model_name == "ARID":
                    model = build_arid(n_features, N_CLASSES)
                elif model_name == "FedHAN":
                    model = build_fedhan(n_features, N_CLASSES)
                elif model_name == "A1 Transformer":
                    model = build_a1_transformer(n_features, N_CLASSES)
                elif model_name == "A2 IAT-DNN":
                    model = build_a2_iat_dnn()
                elif model_name == "A4 GIDS":
                    model = build_a4_gids(n_features, N_CLASSES)
                elif model_name == "A5 GCN-DQN":
                    model = build_a5_gcn_dqn()
                elif model_name == "A3 LLM+RAG":
                    model = build_a3_llm_rag()
                else:
                    raise ValueError(model_name)

                # ---- Data routing -------------------------------------------------------
                # Everyone except A1/A2/A3 gets the standard fold-resampled, class-weighted
                # 26-feature matrix (anomaly detectors ignore class_weight internally).
                # A1/A2/A3 have their own data needs -- see the comment block at the top of
                # each one's cell for why.
                if model_name == "A1 Transformer":
                    # "No imbalance handling" (its whole point): unresampled, unweighted.
                    model.fit(Xtr_scaled, y_train, class_weight=None)
                    proba = model.predict_proba(Xte_scaled)
                    y_eval = y_test
                elif model_name == "A2 IAT-DNN":
                    # Single raw IAT column, sourced from the pre-filter 46-col frame and
                    # put through the SAME fold-internal scaling + resampling as everyone
                    # else (resulting class counts match ytr exactly, since resample_fold
                    # is seeded on the same y_train -- only the feature values differ).
                    iat_scaler = RobustScaler()
                    iat_tr = iat_scaler.fit_transform(X_raw_full.iloc[train_idx][[IAT_COLUMN_NAME]])
                    iat_te = iat_scaler.transform(X_raw_full.iloc[test_idx][[IAT_COLUMN_NAME]])
                    iat_tr_res, y_iat_res = resample_fold(iat_tr, y_train, class_to_idx)
                    model.fit(iat_tr_res, y_iat_res, class_weight=class_weight)
                    proba = model.predict_proba(iat_te)
                    y_eval = y_test
                elif model_name == "A3 LLM+RAG":
                    # Raw (unscaled, named) features + a bounded per-class subsample for
                    # both fine-tuning and evaluation -- see fit_predict_a3's docstring.
                    proba, y_eval = fit_predict_a3(model, X_train_raw, y_train, X_test_raw, y_test)
                else:
                    model.fit(Xtr, ytr, class_weight=class_weight)
                    proba = model.predict_proba(Xte_scaled)
                    y_eval = y_test

                pred = proba.argmax(axis=1)
                metrics = compute_metrics(y_eval, pred, proba)
                elapsed = time.time() - t0

                np.save(f"{PROBA_DIR}/fold{fold}_{model_name.replace(' ', '_')}.npy", proba)
                if model_name == "A3 LLM+RAG":
                    # NOT the shared fold ytrue: A3 evaluates on a bounded subsample, not
                    # the full test split, so it gets its own file rather than silently
                    # mis-aligning with every other model's fold{fold}_ytrue.npy.
                    np.save(f"{PROBA_DIR}/fold{fold}_A3_LLM+RAG_ytrue_subsample.npy", y_eval)
                else:
                    np.save(f"{PROBA_DIR}/fold{fold}_ytrue.npy", y_eval)
                append_result(dict(fold=fold, model=model_name, time_sec=elapsed, **metrics))
                loss_note = ""
                if hasattr(model, "loss_history") and model.loss_history:
                    lh = model.loss_history
                    loss_note = f" loss {lh[0]:.3f}->{lh[-1]:.3f}"
                    if not np.isfinite(lh[-1]):
                        loss_note += "  [!! NON-FINITE LOSS -- TRAINING DIVERGED]"
                    elif lh[-1] > 0.95 * lh[0]:
                        loss_note += "  [!! loss barely moved -- model may not be learning]"
                    # v8: persist the full loss curve to disk for the diagnostic plots
                    # in the Model Performance Visualization section further down.
                    np.save(f"{LOSS_DIR}/fold{fold}_{model_name.replace(' ', '_')}.npy", np.array(lh))
                print(f"  {model_name:22s} f1_macro={metrics['f1_macro']:.4f} "
                      f"acc={metrics['accuracy']:.4f} auc={metrics['auc']:.4f} "
                      f"({elapsed:.1f}s){loss_note}")
                # Sanity check against the majority rate of whatever y_eval actually is
                # (equal to the fold's majority_rate for every model except A3, whose
                # y_eval is a smaller, roughly class-balanced subsample).
                local_majority_rate = np.bincount(y_eval, minlength=N_CLASSES).max() / len(y_eval)
                if metrics["accuracy"] < local_majority_rate:
                    print(f"      [!! SANITY CHECK FAILED] accuracy {metrics['accuracy']:.4f} is BELOW the "
                          f"trivial always-predict-majority rate ({local_majority_rate:.4f}). "
                          f"Treat this as a suspected implementation failure, not a result.")

                if model_name in ("PANE", "CAFE-TF", "ARID", "FedHAN"):
                    fitted_novel[model_name] = model

            except Exception as e:
                print(f"  [FAILED] {model_name}: {e}")
                append_result(dict(fold=fold, model=model_name, time_sec=np.nan,
                                    accuracy=np.nan, precision=np.nan, recall=np.nan,
                                    f1_macro=np.nan, auc=np.nan))

        if not already_done(fold, "Cascaded Ensemble") and len(fitted_novel) == 4:
            t0 = time.time()
            cascade = build_cascade(fitted_novel)
            # v7, changelog D: return_stages=True also gives back which of the three
            # stages (1=PANE, 2=CAFE-TF, 3=ARID+FedHAN fallback) produced each row.
            proba, stage_id = cascade.predict_proba(Xte_scaled, return_stages=True)
            pred = proba.argmax(axis=1)
            metrics = compute_metrics(y_test, pred, proba)
            elapsed = time.time() - t0
            np.save(f"{PROBA_DIR}/fold{fold}_Cascaded_Ensemble.npy", proba)
            np.save(f"{PROBA_DIR}/fold{fold}_Cascaded_Ensemble_stages.npy", stage_id)
            append_result(dict(fold=fold, model="Cascaded Ensemble", time_sec=elapsed, **metrics))

            # v7, changelog D: per-stage resolution rates, logged to their own CSV so
            # per_fold_results.csv's column structure stays identical for every other
            # model (append_result's header is only written once, on the first row).
            n_total = len(stage_id)
            stage_row = dict(
                fold=fold, n_total=n_total,
                n_stage1=int((stage_id == 1).sum()), pct_stage1=float((stage_id == 1).mean() * 100),
                n_stage2=int((stage_id == 2).sum()), pct_stage2=float((stage_id == 2).mean() * 100),
                n_stage3=int((stage_id == 3).sum()), pct_stage3=float((stage_id == 3).mean() * 100),
            )
            stage_csv = f"{OUTPUT_DIR}/cascade_stage_resolution.csv"
            pd.DataFrame([stage_row]).to_csv(stage_csv, mode="a",
                                              header=not os.path.exists(stage_csv), index=False)

            print(f"  {'Cascaded Ensemble':22s} f1_macro={metrics['f1_macro']:.4f} "
                  f"acc={metrics['accuracy']:.4f} auc={metrics['auc']:.4f} ({elapsed:.1f}s) "
                  f"[stage1={stage_row['pct_stage1']:.1f}% stage2={stage_row['pct_stage2']:.1f}% "
                  f"stage3={stage_row['pct_stage3']:.1f}%]")

    print("\nAll folds complete. Results at:", RESULTS_CSV)
    return pd.read_csv(RESULTS_CSV)

print("CV harness (v4) defined. Run `results_df = run_full_evaluation()` next.")


## Statistical Significance — Friedman, Nemenyi, CD Diagram (Figure 8)

In [ ]:
#!pip install scikit-posthocs

In [ ]:
# =====================================================================================
# CELL 15 -- Statistical Significance: Friedman + Nemenyi + CD Diagram (Figure 8)
# =====================================================================================
from scipy import stats
import scikit_posthocs as sp
import matplotlib.pyplot as plt

def run_statistical_analysis(results_df, metric="f1_macro"):
    pivot = results_df.pivot(index="fold", columns="model", values=metric).dropna(axis=1)
    models = pivot.columns.tolist()
    print(f"Friedman test over {len(models)} models across {pivot.shape[0]} folds")

    chi2, p = stats.friedmanchisquare(*[pivot[m].values for m in models])
    df_ = len(models) - 1
    print(f"Friedman chi^2({df_}) = {chi2:.2f}, p = {p:.4g}")

    nemenyi = sp.posthoc_nemenyi_friedman(pivot.values)
    nemenyi.columns = models
    nemenyi.index = models

    avg_ranks = pivot.rank(axis=1, ascending=False).mean(axis=0).sort_values()
    return dict(chi2=chi2, df=df_, p=p, nemenyi=nemenyi, avg_ranks=avg_ranks, n_folds=pivot.shape[0])


def plot_cd_diagram(avg_ranks, n_folds, n_models, alpha=0.05, save_path=None):
    """Critical Difference diagram (Demsar 2006)."""
    k = n_models
    # v9 BUGFIX: this previously hardcoded q_alpha=2.343 for any k>10, which is the
    # correct Nemenyi critical value for k=3, not k=20 -- confirmed independently via
    # scipy's studentized_range distribution (q_alpha(k=20)=3.5438), the same
    # verification already applied to this project's manuscript several rounds ago.
    # The hardcoded value silently produced CD=8.767 instead of the correct CD=13.26
    # for a 20-method comparison, understating the critical difference by ~40% and
    # overstating how many methods appear significantly different. Computed exactly
    # for the actual k and alpha now, not looked up from a 2-entry table.
    from scipy.stats import studentized_range
    q_alpha = studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2)
    cd = q_alpha * np.sqrt(k * (k + 1) / (6.0 * n_folds))

    names = avg_ranks.index.tolist()
    ranks = avg_ranks.values
    fig, ax = plt.subplots(figsize=(9, 0.35 * len(names) + 1.5), dpi=200)
    ax.set_xlim(min(ranks) - 1, max(ranks) + 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    y0 = 0.85
    ax.plot([min(ranks), min(ranks) + cd], [y0, y0], color="black", lw=1.5)
    ax.text(min(ranks) + cd / 2, y0 + 0.05, f"CD = {cd:.3f}", ha="center", fontsize=9)

    for i, (name, r) in enumerate(zip(names, ranks)):
        y = 0.65 - i * (0.6 / len(names))
        ax.plot([r, r], [y - 0.01, y + 0.01], color="black")
        ax.text(r, y - 0.03, f"{name}\n({r:.2f})", ha="center", va="top", fontsize=7)

    plt.title(f"Fig. 8 \u2014 Critical Difference Diagram ({n_models} models, {n_folds}-fold, "
              f"Nemenyi \u03b1={alpha})", fontsize=10)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        plt.savefig(save_path.replace(".png", ".svg"), bbox_inches="tight")
    plt.show()

print("Statistical analysis + CD diagram functions defined.")


## ROC Curves (Figure 5)

In [ ]:
# =====================================================================================
# ROC Curves (Figure 5), Macro One-vs-Rest, averaged across folds
# class imbalance a micro-averaged curve is implicitly dominated by the majority
# classes and can differ substantially from a true macro average. Now computes one
# ROC curve per class, interpolates each onto a shared FPR grid, and averages across
# classes -- i.e. genuinely macro -- before averaging across folds as before. As a
# correctness check, the AUC values this now reports should land close to (within a
# few thousandths of) the "auc" column already in per_fold_results.csv, since that
# column has always used sklearn's correct macro-OvR roc_auc_score (Cell 7); if they
# don't, something is still wrong and should not be trusted blindly.)
# =====================================================================================
from sklearn.preprocessing import label_binarize

def plot_roc_curves(model_names, n_folds=N_FOLDS, n_classes=N_CLASSES, save_path=None):
    fig, ax = plt.subplots(figsize=(7, 6), dpi=200)
    mean_fpr = np.linspace(0, 1, 200)

    for model_name in model_names:
        fold_tprs = []  # one genuinely macro-averaged TPR curve per fold
        for fold in range(1, n_folds + 1):
            proba_path = f"{PROBA_DIR}/fold{fold}_{model_name.replace(' ', '_')}.npy"
            ytrue_path = f"{PROBA_DIR}/fold{fold}_ytrue.npy"
            if not (os.path.exists(proba_path) and os.path.exists(ytrue_path)):
                continue
            proba = np.load(proba_path)
            y_true = np.load(ytrue_path)
            y_bin = label_binarize(y_true, classes=list(range(n_classes)))

            class_tprs = []
            for k in range(n_classes):
                if y_bin[:, k].sum() == 0:
                    continue  # no positive examples for this class in this fold
                fpr_k, tpr_k, _ = roc_curve(y_bin[:, k], proba[:, k])
                class_tprs.append(np.interp(mean_fpr, fpr_k, tpr_k))
            if not class_tprs:
                continue
            macro_tpr_this_fold = np.mean(class_tprs, axis=0)  # macro step: avg over classes
            macro_tpr_this_fold[0] = 0.0
            fold_tprs.append(macro_tpr_this_fold)

        if not fold_tprs:
            print(f"  [no saved probabilities for {model_name}, skipping]")
            continue
        mean_tpr = np.mean(fold_tprs, axis=0)  # then average over folds, as before
        mean_tpr[0] = 0.0
        trapezoid_fn = getattr(np, "trapezoid", None) or np.trapz
        auc_val = trapezoid_fn(mean_tpr, mean_fpr)
        ax.plot(mean_fpr, mean_tpr, label=f"{model_name} (AUC={auc_val:.4f})", linewidth=1.6)

    ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=0.8)
    ax.set_xlim(0, 0.25)
    ax.set_ylim(0.93, 1.0)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Fig. 5 \u2014 Receiver Operating Characteristic Curves\n(Macro One-vs-Rest, mean over folds)")
    ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        plt.savefig(save_path.replace(".png", ".svg"), bbox_inches="tight")
    plt.show()

print("ROC curve function defined (macro-averaged).")


## Run Everything
Fast pass first, confirm no errors, then the real run.

In [ ]:
# =====================================================================================
# CELL 17 -- Run Everything
# =====================================================================================
results_df = run_full_evaluation()

stats_result = run_statistical_analysis(results_df, metric="f1_macro")
print(f"\nFriedman chi^2({stats_result['df']}) = {stats_result['chi2']:.2f}, p = {stats_result['p']:.4g}")
print("\nAverage ranks (lower = better):")
print(stats_result["avg_ranks"])

stats_result["nemenyi"].to_csv(f"{OUTPUT_DIR}/nemenyi_pvalues.csv")
stats_result["avg_ranks"].to_csv(f"{OUTPUT_DIR}/avg_ranks.csv")

plot_cd_diagram(stats_result["avg_ranks"], stats_result["n_folds"],
                 len(stats_result["avg_ranks"]),
                 save_path=f"{OUTPUT_DIR}/figures/Figure8_CD_diagram.png")

roc_models = ["Bi-LSTM", "A1 Transformer", "A2 IAT-DNN", "A4 GIDS", "A5 GCN-DQN",
              "PANE", "CAFE-TF", "ARID", "FedHAN", "Cascaded Ensemble"]
# A3 LLM+RAG is deliberately left out here: it evaluates on a bounded per-class
# subsample (fold{fold}_A3_LLM+RAG_ytrue_subsample.npy), not the full fold's
# fold{fold}_ytrue.npy that plot_roc_curves assumes every model shares.
plot_roc_curves(roc_models, save_path=f"{OUTPUT_DIR}/figures/Figure5_ROC_curves.png")

summary = results_df.groupby("model")[["accuracy", "precision", "recall", "f1_macro", "auc", "time_sec"]].agg(["mean", "std"])
summary.to_csv(f"{OUTPUT_DIR}/table_III_summary.csv")
print("\nSaved: per_fold_results.csv, table_III_summary.csv, nemenyi_pvalues.csv, "
      "avg_ranks.csv, figures/Figure8_CD_diagram.png(+.svg), figures/Figure5_ROC_curves.png(+.svg)")


## Model Performance Visualization Suite (v8)

Everything below reads already-computed outputs -- `per_fold_results.csv`, `cascade_stage_resolution.csv`, the saved per-fold `.npy` prediction/probability arrays, and (if the v8 main-loop patch below has run) saved loss-history arrays. Nothing in this section re-fits or re-evaluates any model, so it is safe and fast to re-run on its own after `run_full_evaluation()` completes, or partway through if you want to check progress.

In [ ]:
# =====================================================================================
# Performance visualization setup
# =====================================================================================
results_df = pd.read_csv(RESULTS_CSV)

_PROPOSED = {"PANE", "CAFE-TF", "ARID", "FedHAN", "Cascaded Ensemble"}
_LITERATURE = {"A1 Transformer", "A2 IAT-DNN", "A4 GIDS", "A5 GCN-DQN"}
_DISPLAY_NAME = {"A1 Transformer": "Transformer", "A2 IAT-DNN": "IAT-DNN",
                 "A4 GIDS": "GIDS", "A5 GCN-DQN": "GCN-DQN"}
_CATEGORY_COLOR = {"Proposed": "#2E7D5B", "Literature": "#E08E45", "Baseline": "#8C9BAB"}

def _category_of(model_name):
    if model_name in _PROPOSED:
        return "Proposed"
    if model_name in _LITERATURE:
        return "Literature"
    return "Baseline"

def _display_name_of(model_name):
    return _DISPLAY_NAME.get(model_name, model_name)

summary_df = results_df.groupby("model")[["accuracy", "precision", "recall", "f1_macro", "auc", "time_sec"]].agg(["mean", "std"])
summary_df.columns = ["_".join(c) for c in summary_df.columns]
summary_df["category"] = [_category_of(m) for m in summary_df.index]
summary_df["display_name"] = [_display_name_of(m) for m in summary_df.index]
summary_df = summary_df.sort_values("f1_macro_mean", ascending=False)

FIG_DIR = f"{OUTPUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

print(f"{len(summary_df)} models with results so far. Summary (sorted by macro-F1):")
print(summary_df[["display_name", "category", "f1_macro_mean", "f1_macro_std",
                   "accuracy_mean", "auc_mean"]].round(4))


In [ ]:
# ---- Fig P1: Macro-F1 comparison, all evaluated models ----
fig, ax = plt.subplots(figsize=(8, 0.35 * len(summary_df) + 1.5))
plot_df = summary_df.sort_values("f1_macro_mean", ascending=True)
y_pos = np.arange(len(plot_df))
colors = [_CATEGORY_COLOR[c] for c in plot_df["category"]]
ax.barh(y_pos, plot_df["f1_macro_mean"], xerr=plot_df["f1_macro_std"], color=colors,
        edgecolor="black", linewidth=0.4, capsize=2)
ax.set_yticks(y_pos); ax.set_yticklabels(plot_df["display_name"], fontsize=8)
ax.set_xlabel("Macro-F1 (mean +/- std over folds)")
ax.set_title("Macro-F1 Across All Evaluated Methods", fontsize=11, fontweight="bold")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=v, edgecolor="black", label=k) for k, v in _CATEGORY_COLOR.items()],
          loc="lower right", fontsize=8)
ax.grid(axis="x", linestyle=":", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
fig_path = f"{FIG_DIR}/Perf_macroF1_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig P2: accuracy vs macro-F1 (imbalance-robustness view) ----
fig, ax = plt.subplots(figsize=(8, 7))
for cat, color in _CATEGORY_COLOR.items():
    sub = summary_df[summary_df["category"] == cat]
    if len(sub) == 0:
        continue
    ax.errorbar(sub["accuracy_mean"], sub["f1_macro_mean"], xerr=sub["accuracy_std"],
                yerr=sub["f1_macro_std"], fmt="o", ms=5, color=color, ecolor=color,
                elinewidth=0.5, capsize=1.5, alpha=0.85, label=cat,
                markeredgecolor="black", markeredgewidth=0.3)
for name, row in summary_df.iterrows():
    ax.annotate(row["display_name"], (row["accuracy_mean"], row["f1_macro_mean"]),
                fontsize=6, xytext=(4, 3), textcoords="offset points")
if "y_viz" in dir():
    true_majority_rate = pd.Series(y_viz).value_counts(normalize=True).max()
    ax.axvline(true_majority_rate, color="grey", linestyle="--", linewidth=0.8)
    ax.text(true_majority_rate + 0.01, ax.get_ylim()[0] + 0.02,
            f"always-predict-majority\naccuracy ({true_majority_rate:.3f})", fontsize=7, color="grey")
ax.set_xlabel("Accuracy (mean over folds)"); ax.set_ylabel("Macro-F1 (mean over folds)")
ax.set_title("Accuracy vs. Macro-F1: the Imbalance-Robustness Gap", fontsize=11, fontweight="bold")
ax.legend(loc="best", fontsize=8)
ax.grid(linestyle=":", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
fig_path = f"{FIG_DIR}/Perf_accuracy_vs_macroF1.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig P3: Macro one-vs-rest AUC comparison ----
auc_df = summary_df.dropna(subset=["auc_mean"]).sort_values("auc_mean", ascending=True)
fig, ax = plt.subplots(figsize=(8, 0.35 * len(auc_df) + 1.5))
colors = [_CATEGORY_COLOR[c] for c in auc_df["category"]]
ax.barh(np.arange(len(auc_df)), auc_df["auc_mean"], xerr=auc_df["auc_std"], color=colors,
        edgecolor="black", linewidth=0.4, capsize=2)
ax.set_yticks(np.arange(len(auc_df))); ax.set_yticklabels(auc_df["display_name"], fontsize=8)
ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.6)
ax.set_xlabel("Macro One-vs-Rest ROC-AUC (mean +/- std)")
ax.set_title(f"Macro AUC Comparison ({len(auc_df)}/{len(summary_df)} models with defined AUC)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
fig_path = f"{FIG_DIR}/Perf_macroAUC_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")
n_undefined = len(summary_df) - len(auc_df)
if n_undefined:
    print(f"  [note] {n_undefined} model(s) have undefined (NaN) AUC -- degenerate probability "
          f"output. Check the confusion-matrix and per-class-F1 cells below for why.")


In [ ]:
# ---- Fig P4: wall-clock time vs macro-F1 ----
fig, ax = plt.subplots(figsize=(8, 6.5))
for cat, color in _CATEGORY_COLOR.items():
    sub = summary_df[summary_df["category"] == cat]
    if len(sub) == 0:
        continue
    ax.errorbar(sub["time_sec_mean"], sub["f1_macro_mean"], yerr=sub["f1_macro_std"],
                fmt="o", ms=5.5, color=color, ecolor=color, elinewidth=0.5, capsize=1.5,
                alpha=0.85, label=cat, markeredgecolor="black", markeredgewidth=0.3)
for name, row in summary_df.iterrows():
    ax.annotate(row["display_name"], (row["time_sec_mean"], row["f1_macro_mean"]),
                fontsize=6, xytext=(5, 3), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Wall-clock time per fold, log scale (s)\n"
              "(Cascaded Ensemble is inference-only over its 4 already-fitted components)")
ax.set_ylabel("Macro-F1 (mean over folds)")
ax.set_title("Computational Cost vs. Macro-F1", fontsize=11, fontweight="bold")
ax.legend(loc="best", fontsize=8)
ax.grid(linestyle=":", alpha=0.5, which="both"); ax.set_axisbelow(True)
plt.tight_layout()
fig_path = f"{FIG_DIR}/Perf_time_vs_macroF1.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig P5: per-fold macro-F1 distribution (box plot) -- stability vs instability ----
order = summary_df.sort_values("f1_macro_mean", ascending=False).index.tolist()
box_data = [results_df.loc[results_df["model"] == m, "f1_macro"].dropna().values for m in order]
labels = [_display_name_of(m) for m in order]

fig, ax = plt.subplots(figsize=(max(10, 0.5 * len(order)), 6))
bp = ax.boxplot(box_data, patch_artist=True, showmeans=True, widths=0.6)
for patch, m in zip(bp["boxes"], order):
    patch.set_facecolor(_CATEGORY_COLOR[_category_of(m)])
    patch.set_alpha(0.7)
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=75, fontsize=7)
ax.set_ylabel("Macro-F1 per fold")
ax.set_title("Per-Fold Macro-F1 Distribution\n"
             "(a tight cluster = consistently stable; a wide box or scattered points = "
             "fold-dependent, e.g. unstable federated training)", fontsize=10, fontweight="bold")
ax.grid(axis="y", linestyle=":", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
fig_path = f"{FIG_DIR}/Perf_fold_variance_boxplot.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig P6: confusion matrices for a curated subset of models ----
# Uses the per-fold saved .npy prediction arrays the main loop already writes for every
# model -- pools predictions across all 5 folds' test splits into one confusion matrix
# per model. No re-inference needed.
from sklearn.metrics import confusion_matrix

_confmat_models = ["CAFE-TF", "Cascaded Ensemble", "PANE", "ARID", "FedHAN", "CNN-1D",
                    "A1 Transformer", "VAE Detector"]
_confmat_models = [m for m in _confmat_models if m in results_df["model"].unique()]

n_cols = 4
n_rows = int(np.ceil(len(_confmat_models) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax_i, model_name in enumerate(_confmat_models):
    ax = axes[ax_i]
    all_true, all_pred = [], []
    for fold in range(1, N_FOLDS + 1):
        proba_path = f"{PROBA_DIR}/fold{fold}_{model_name.replace(' ', '_')}.npy"
        ytrue_path = f"{PROBA_DIR}/fold{fold}_ytrue.npy"
        if not (os.path.exists(proba_path) and os.path.exists(ytrue_path)):
            continue
        proba = np.load(proba_path)
        all_true.append(np.load(ytrue_path))
        all_pred.append(proba.argmax(axis=1))
    if not all_true:
        ax.set_visible(False)
        continue
    y_true_cat = np.concatenate(all_true)
    y_pred_cat = np.concatenate(all_pred)
    cm = confusion_matrix(y_true_cat, y_pred_cat, labels=list(range(N_CLASSES)), normalize="true")
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_ORDER, rotation=90, fontsize=6)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_ORDER, fontsize=6)
    ax.set_title(_display_name_of(model_name), fontsize=9, fontweight="bold")

for ax_i in range(len(_confmat_models), len(axes)):
    axes[ax_i].set_visible(False)

fig.suptitle("Row-Normalized Confusion Matrices (all folds pooled), Curated Subset",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
fig_path = f"{FIG_DIR}/Perf_confusion_matrices.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig P7: per-class F1 heatmap, every evaluated model x 8 classes ----
from sklearn.metrics import f1_score

all_models_list = sorted(results_df["model"].unique())
per_class_f1 = np.full((len(all_models_list), N_CLASSES), np.nan)

for mi, model_name in enumerate(all_models_list):
    all_true, all_pred = [], []
    for fold in range(1, N_FOLDS + 1):
        proba_path = f"{PROBA_DIR}/fold{fold}_{model_name.replace(' ', '_')}.npy"
        ytrue_path = f"{PROBA_DIR}/fold{fold}_ytrue.npy"
        if not (os.path.exists(proba_path) and os.path.exists(ytrue_path)):
            continue
        proba = np.load(proba_path)
        all_true.append(np.load(ytrue_path))
        all_pred.append(proba.argmax(axis=1))
    if not all_true:
        continue
    y_true_cat = np.concatenate(all_true)
    y_pred_cat = np.concatenate(all_pred)
    per_class_f1[mi] = f1_score(y_true_cat, y_pred_cat, labels=list(range(N_CLASSES)),
                                 average=None, zero_division=0)

with warnings.catch_warnings():  # a model with zero saved files leaves an
    warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN row; harmless
    row_means = np.nanmean(per_class_f1, axis=1)
# argsort (not nanargsort -- that function does not exist in numpy) already sorts
# NaN to the end regardless of sign, so this gives descending-by-F1 with any
# no-data models correctly pushed to the bottom rather than crashing.
order_idx = np.argsort(-row_means)
per_class_f1_sorted = per_class_f1[order_idx]
model_labels_sorted = [_display_name_of(all_models_list[i]) for i in order_idx]

fig, ax = plt.subplots(figsize=(8, 0.35 * len(all_models_list) + 2))
im = ax.imshow(per_class_f1_sorted, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_ORDER, rotation=45, ha="right")
ax.set_yticks(range(len(model_labels_sorted))); ax.set_yticklabels(model_labels_sorted, fontsize=7)
for i in range(per_class_f1_sorted.shape[0]):
    for j in range(per_class_f1_sorted.shape[1]):
        v = per_class_f1_sorted[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5.5,
                    color="white" if (v < 0.35 or v > 0.75) else "black")
plt.colorbar(im, ax=ax, label="Per-class F1", shrink=0.8)
ax.set_title("Per-Class F1 by Model -- which classes does each model actually get right?",
             fontsize=10, fontweight="bold")
plt.tight_layout()
fig_path = f"{FIG_DIR}/Perf_per_class_F1_heatmap.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
# ---- Fig P8: training loss curves (needs the v8 main-loop loss-history patch below) ----
loss_dir = f"{OUTPUT_DIR}/loss_history"
if os.path.isdir(loss_dir) and os.listdir(loss_dir):
    _highlight = [m for m in ["CNN-1D", "FedHAN", "MLP", "CAFE-TF"] if m in results_df["model"].unique()]
    fig, axes = plt.subplots(1, len(_highlight), figsize=(4.2 * len(_highlight), 4))
    axes = np.array(axes).reshape(-1) if len(_highlight) > 1 else np.array([axes])
    for ax, model_name in zip(axes, _highlight):
        any_plotted = False
        for fold in range(1, N_FOLDS + 1):
            p = f"{loss_dir}/fold{fold}_{model_name.replace(' ', '_')}.npy"
            if os.path.exists(p):
                lh = np.load(p)
                ax.plot(lh, label=f"fold {fold}", linewidth=1.2)
                any_plotted = True
        ax.set_title(model_name, fontsize=9, fontweight="bold")
        ax.set_xlabel("epoch"); ax.set_ylabel("training loss")
        if any_plotted:
            ax.legend(fontsize=6)
        else:
            ax.text(0.5, 0.5, "no saved loss history\nfor this model", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color="grey")
    fig.suptitle("Training Loss Trajectories -- direct diagnostic for CNN-1D / FedHAN's\n"
                 "still-unresolved instability (changelog item C)", fontsize=11, fontweight="bold", y=1.05)
    plt.tight_layout()
    fig_path = f"{FIG_DIR}/Perf_loss_curves.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_path}")
else:
    print(f"  [skip] {loss_dir} not found or empty -- this needs the v8 loss-history-saving "
          f"patch in the main loop cell to have run first.")


In [ ]:
# ---- Fig P9: cascade per-stage resolution rate (needs the v7 stage-tracking patch) ----
stage_csv = f"{OUTPUT_DIR}/cascade_stage_resolution.csv"
if os.path.exists(stage_csv):
    stage_df = pd.read_csv(stage_csv)
    fig, ax = plt.subplots(figsize=(7, 5))
    x = stage_df["fold"].values
    ax.bar(x, stage_df["pct_stage1"], label="Stage 1 (PANE)", color="#2E7D5B")
    ax.bar(x, stage_df["pct_stage2"], bottom=stage_df["pct_stage1"],
           label="Stage 2 (CAFE-TF)", color="#E08E45")
    ax.bar(x, stage_df["pct_stage3"], bottom=stage_df["pct_stage1"] + stage_df["pct_stage2"],
           label="Stage 3 (ARID+FedHAN fallback)", color="#8C9BAB")
    ax.set_xlabel("Fold"); ax.set_xticks(x)
    ax.set_ylabel("% of test samples resolved at this stage")
    ax.set_title("Cascade Per-Stage Resolution Rate by Fold", fontsize=11, fontweight="bold")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
    plt.tight_layout()
    fig_path = f"{FIG_DIR}/Perf_cascade_stage_resolution.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_path}")
    print(stage_df[["pct_stage1", "pct_stage2", "pct_stage3"]].agg(["mean", "std"]).round(2))
else:
    print(f"  [skip] {stage_csv} not found -- needs the v7 cascade stage-tracking patch "
          f"to have run first.")


In [ ]:
# ---- Fig P10: Cascaded Ensemble vs its own strongest component (CAFE-TF), fold by fold ----
cafe_f1 = results_df.loc[results_df["model"] == "CAFE-TF"].sort_values("fold")["f1_macro"].values
casc_f1 = results_df.loc[results_df["model"] == "Cascaded Ensemble"].sort_values("fold")["f1_macro"].values
if len(cafe_f1) and len(casc_f1) and len(cafe_f1) == len(casc_f1):
    folds_x = np.arange(1, len(cafe_f1) + 1)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(folds_x, cafe_f1, "o-", label="CAFE-TF (alone)", color="#2E7D5B", linewidth=1.8)
    ax.plot(folds_x, casc_f1, "s--", label="Cascaded Ensemble", color="#5B4E8C", linewidth=1.8)
    ax.set_xticks(folds_x); ax.set_xlabel("Fold"); ax.set_ylabel("Macro-F1")
    ax.set_title("Does the Cascade Beat Its Own Strongest Component?", fontsize=11, fontweight="bold")
    ax.legend()
    ax.grid(linestyle=":", alpha=0.5)
    plt.tight_layout()
    fig_path = f"{FIG_DIR}/Perf_cascade_vs_cafetf.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_path}")
    print(f"CAFE-TF wins {int((cafe_f1 > casc_f1).sum())}/{len(cafe_f1)} folds on macro-F1.")
else:
    print("  [skip] need both CAFE-TF and Cascaded Ensemble results for all folds to compare.")
